<a href="https://colab.research.google.com/github/Balaji-jrf/JRF-Work-Repo/blob/main/GraphRAG_Implementaion2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
data = "/content/1917_The Antoquities of Bidar _book_ocr.json"

In [2]:
import json
import os
import pandas as pd

# 1. Load the OCR JSON file (prompts upload in Colab if not already in /content/)
json_filename = data

if not os.path.exists(json_filename):
    from google.colab import files
    uploaded = files.upload()
    json_filename = list(uploaded.keys())[0]

with open(json_filename, "r", encoding="utf-8") as f:
    ocr_data = json.load(f)

total_pages = ocr_data.get("total_pages", 70)
doc_id = ocr_data.get("document_id", "unknown_doc")
source_pdf = ocr_data.get("filename", "1917_The Antoquities of Bidar _book.pdf")

print(f"Loaded: {source_pdf} | Total Pages: {total_pages} | Elements: {ocr_data.get('element_count')}")

# 2. Reconstruct text page-by-page (Pages 1 to 70)
# Prefer 'layout' blocks (excluding noisy standalone page_number tags) and sort top-to-bottom
raw_blocks = ocr_data.get("layout") or ocr_data.get("elements", [])

page_buckets = {p: [] for p in range(1, total_pages + 1)}
for block in raw_blocks:
    page_num = int(block.get("page", 0))
    block_type = block.get("type", "text")
    text = block.get("text", "").strip()
    bbox = block.get("bbox", [0, 0, 0, 0])

    if page_num in page_buckets and text and block_type != "page_number":
        page_buckets[page_num].append((bbox[1], bbox[0], text))

page_texts = {}
for page_num in range(1, total_pages + 1):
    # Sort by vertical position (y1), then horizontal position (x1)
    sorted_items = sorted(page_buckets[page_num], key=lambda item: (round(item[0], 3), round(item[1], 3)))
    page_texts[page_num] = "\n".join(item[2] for item in sorted_items)

# 3. Group into fixed 5-page chunks (14 chunks total for 70 pages)
CHUNK_SIZE = 5
chunks = []

for idx, start_page in enumerate(range(1, total_pages + 1, CHUNK_SIZE), start=1):
    end_page = min(start_page + CHUNK_SIZE - 1, total_pages)
    page_range = list(range(start_page, end_page + 1))

    # Keep individual page texts inside the chunk for Step 2 entity-to-page linking
    chunk_pages = {p: page_texts[p] for p in page_range}
    active_pages = [p for p in page_range if chunk_pages[p].strip()]

    formatted_sections = []
    for p in page_range:
        content = chunk_pages[p] if chunk_pages[p].strip() else "[Blank / Uncaptioned Plate Page]"
        formatted_sections.append(f"--- [Page {p}] ---\n{content}")

    combined_text = "\n\n".join(formatted_sections)

    chunks.append({
        "chunk_id": f"chunk_{idx:02d}",
        "chunk_index": idx,
        "document_id": doc_id,
        "source_file": source_pdf,
        "start_page": start_page,
        "end_page": end_page,
        "page_numbers": page_range,
        "active_pages": active_pages,
        "char_count": len(combined_text),
        "pages": chunk_pages,
        "text": combined_text
    })

# Save intermediate output for Step 2
with open("step1_page_chunks.json", "w", encoding="utf-8") as out_f:
    json.dump(chunks, out_f, indent=2, ensure_ascii=False)

# 4. Display a clean summary table of all 14 chunks
summary_df = pd.DataFrame([
    {
        "Chunk ID": c["chunk_id"],
        "Page Window": f"Pages {c['start_page']}–{c['end_page']}",
        "Pages With Text": str(c["active_pages"]),
        "Total Characters": c["char_count"],
        "Preview": c["text"].replace("\n", " ")[:95] + "..."
    }
    for c in chunks
])

display(summary_df)

# Inspect Chunk 2 (Pages 6-10) in detail
print("\n=== SAMPLE INSPECTION: chunk_02 (Pages 6–10) ===\n")
print(chunks[1]["text"][:1500])

Saving 1917_The Antoquities of Bidar _book_ocr.json to 1917_The Antoquities of Bidar _book_ocr.json
Loaded: 1917_The Antoquities of Bidar _book.pdf | Total Pages: 70 | Elements: 1221


,Chunk ID,Page Window,Pages With Text,Total Characters,Preview
0,chunk_01,Pages 1–5,"[1, 2, 5]",329,--- [Page 1] --- AND LABOR OF BIDAR --- [Page...
1,chunk_02,Pages 6–10,"[6, 7, 8, 9, 10]",4341,--- [Page 6] --- - DATE MAXXXXX --- [Page 7] ...
2,chunk_03,Pages 11–15,"[11, 12, 13, 14, 15]",10505,--- [Page 11] --- a prisoner to the Imperial C...
3,chunk_04,Pages 16–20,"[16, 17, 18, 19, 20]",10576,--- [Page 16] --- Jilas who jugu 3. TO Jail s'...
4,chunk_05,Pages 21–25,"[21, 22, 23, 24, 25]",9109,--- [Page 21] --- yet the ruins of its noble m...
5,chunk_06,Pages 26–30,"[26, 27, 28, 29, 30]",10188,--- [Page 26] --- the officers of the househol...
6,chunk_07,Pages 31–35,"[31, 32, 33, 34, 35]",7334,--- [Page 31] --- Barid Shahi tombs. The next ...
7,chunk_08,Pages 36–40,"[36, 37, 39]",1295,"--- [Page 36] --- BIBLIOGRAPHY. (I) Bakhshi, N..."
8,chunk_09,Pages 41–45,"[41, 43, 45]",351,--- [Page 41] --- TII PARAE a. BIDAR FORT: LAR...
9,chunk_10,Pages 46–50,"[47, 49]",263,--- [Page 46] --- [Blank / Uncaptioned Plate P...



=== SAMPLE INSPECTION: chunk_02 (Pages 6–10) ===

--- [Page 6] ---
- DATE MAXXXXX

--- [Page 7] ---
THE ANTIQUITIES OF BIDAR
G. YAZDANI
S
ASIAN EDUCATIONAL SERVICES NEW DELHI MADRAS 1992

--- [Page 8] ---
ASIAN EDUCATIONAL SERVICES.
* C-2/15, S.D.A. NEW DELHI-110016
* 5 SRIPURAM FIRST STREET, MADRAS-600014.
AES a 285-
First Published:1917 AES Reprint: 1992 ISBN: 81-206-0501-2
Published by J. Jetley for ASIAN EDUCATIONAL SERVICES C-2/15, SDA New Delhi- 10016 Processed by APEX PUBLICATION SERVICES New Delhi-110016 Printed at Gaytri Offset Printers A-66, S. No. 2 Noida, Distt. Ghaziabad (U.P.)

--- [Page 9] ---
THE ANTIQUITIES OF BIDAR.
Should my heart ache, my remedy is this ; A cup of wine, and then I sip of bliss." Inscription over the tomb of Ahmad Shah Bahmani, the founder of modern Bidar.
HE town of Bidar stands on a beautiful plateau 2,330 feet above the sea-level; its situation is pic- turesque and its climate bracing: it is a place over which the spirit of romance seems to have 

In [3]:
#Print all chunks data one by one
for chunk in chunks:
  print("\n=== CHUNK DETAILS ===\n")
  print(f"Chunk ID: {chunk['chunk_id']}")
  print(f"Start Page: {chunk['start_page']}")
  print(f"End Page: {chunk['end_page']}")
  print(f"Active Pages: {chunk['active_pages']}")
  print(f"Total Characters: {chunk['char_count']}")
  print("\n=== CHUNK TEXT ===\n")
  print(chunk['text'])
  print("\n--- END OF CHUNK ---\n")



=== CHUNK DETAILS ===

Chunk ID: chunk_01
Start Page: 1
End Page: 5
Active Pages: [1, 2, 5]
Total Characters: 329

=== CHUNK TEXT ===

--- [Page 1] ---
AND LABOR OF BIDAR

--- [Page 2] ---
Digitized by the Internet Archive in 2019 with funding from Public.Resource.Org
https://archive.org/details/antoquitiesofbid00gyaz

--- [Page 3] ---
[Blank / Uncaptioned Plate Page]

--- [Page 4] ---
[Blank / Uncaptioned Plate Page]

--- [Page 5] ---
THE ANTIQUITIES OF BIDAR

--- END OF CHUNK ---


=== CHUNK DETAILS ===

Chunk ID: chunk_02
Start Page: 6
End Page: 10
Active Pages: [6, 7, 8, 9, 10]
Total Characters: 4341

=== CHUNK TEXT ===

--- [Page 6] ---
- DATE MAXXXXX

--- [Page 7] ---
THE ANTIQUITIES OF BIDAR
G. YAZDANI
S
ASIAN EDUCATIONAL SERVICES NEW DELHI MADRAS 1992

--- [Page 8] ---
ASIAN EDUCATIONAL SERVICES.
* C-2/15, S.D.A. NEW DELHI-110016
* 5 SRIPURAM FIRST STREET, MADRAS-600014.
AES a 285-
First Published:1917 AES Reprint: 1992 ISBN: 81-206-0501-2
Published by J. Jetley for ASIAN EDUC

In [4]:
import json
import re
from collections import defaultdict
import pandas as pd

# 1. Load Step 1 output
with open("step1_page_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# 2. Year Extraction Engine (supports A.D., A.H., ranges like 1492-1619, and contextual 4-digit years)
# Pattern A: Explicit era marker after or before year (e.g., "1432 A.D.", "1082 A.H.", "A.D. 1347", "A.H. 832")
EXPLICIT_ERA_RE = re.compile(
    r"(?:\b(A\.?\s*D\.?|A\.?\s*H\.?|C\.?\s*E\.?)\s*(\d{3,4})\b)|"
    r"(?:\b(\d{3,4})\s*(A\.?\s*D\.?|A\.?\s*H\.?|C\.?\s*E\.?)\b)",
    re.IGNORECASE,
)

# Pattern B: Year ranges (e.g., "1492-1619" or "1492–1619")
RANGE_RE = re.compile(r"\b(1[0-9]{3}|20[0-2][0-9])\s*[-–—]\s*(1[0-9]{3}|20[0-2][0-9])\b")

# Pattern C: Standalone 4-digit historical/publication years (1000–2026) not followed by measurement/currency units
STANDALONE_YEAR_RE = re.compile(
    r"\b(1[0-9]{3}|20[0-2][0-9])\b(?!\s*(?:ft|feet|inch|inches|yard|yards|mile|miles|rs|rupees|pp|pages|vol|fig|pl|plate)\b)",
    re.IGNORECASE,
)

def normalize_calendar(raw_era: str) -> str:
    cleaned = re.sub(r"[\s.]", "", raw_era).upper()
    return "AH" if cleaned == "AH" else "AD"

def extract_years_from_page(text: str):
    """
    Returns a list of dicts:
    [{'entity_id': 'year_AD_1432', 'name': '1432 A.D.', 'year_value': 1432, 'calendar': 'AD', 'context': '...'}]
    """
    if not text or not text.strip():
        return []

    found = []
    occupied_spans = []

    def get_snippet(start, end, window=65):
        s = max(0, start - window)
        e = min(len(text), end + window)
        return " ".join(text[s:e].split())

    def overlaps(start, end):
        return any(not (end <= s or start >= e) for s, e in occupied_spans)

    # Pass 1: Explicit A.D. / A.H. dates (highest precision, captures 3-digit Hijri years like 832 A.H.)
    for m in EXPLICIT_ERA_RE.finditer(text):
        if m.group(1) and m.group(2):
            cal = normalize_calendar(m.group(1))
            yr = int(m.group(2))
        else:
            yr = int(m.group(3))
            cal = normalize_calendar(m.group(4))

        # Sanity bounds: Hijri 1..1450, A.D. 500..2026
        if (cal == "AH" and 1 <= yr <= 1450) or (cal == "AD" and 500 <= yr <= 2026):
            occupied_spans.append(m.span())
            found.append({
                "entity_id": f"year_{cal}_{yr}",
                "name": f"{yr} {'A.H.' if cal == 'AH' else 'A.D.'}",
                "year_value": yr,
                "calendar": cal,
                "context": get_snippet(*m.span()),
            })

    # Pass 2: Hyphenated year ranges (e.g., 1492-1619)
    for m in RANGE_RE.finditer(text):
        if overlaps(*m.span()):
            continue
        yr1, yr2 = int(m.group(1)), int(m.group(2))
        if 1000 <= yr1 <= 2026 and 1000 <= yr2 <= 2026 and yr1 <= yr2:
            occupied_spans.append(m.span())
            snippet = get_snippet(*m.span())
            for yr in (yr1, yr2):
                found.append({
                    "entity_id": f"year_AD_{yr}",
                    "name": f"{yr} A.D.",
                    "year_value": yr,
                    "calendar": "AD",
                    "context": snippet,
                })

    # Pass 3: Standalone 4-digit years (defaulting to A.D. in this text)
    for m in STANDALONE_YEAR_RE.finditer(text):
        if overlaps(*m.span()):
            continue
        yr = int(m.group(1))
        if 1000 <= yr <= 2026:
            occupied_spans.append(m.span())
            found.append({
                "entity_id": f"year_AD_{yr}",
                "name": f"{yr} A.D.",
                "year_value": yr,
                "calendar": "AD",
                "context": get_snippet(*m.span()),
            })

    return found

# 3. Build Graph Nodes & Relationships
nodes = {
    "Document": [],
    "Chunk": [],
    "Page": [],
    "ENTITIES": [],
}
relationships = []

doc_id = chunks[0]["document_id"]
source_file = chunks[0]["source_file"]

nodes["Document"].append({
    "node_id": doc_id,
    "label": "Document",
    "title": source_file,
    "total_pages": 70,
})

entity_registry = {}  # entity_id -> entity node dict
entity_chunk_tracker = defaultdict(lambda: {"count": 0, "pages": set()})

for chunk in chunks:
    c_id = chunk["chunk_id"]
    nodes["Chunk"].append({
        "node_id": c_id,
        "label": "Chunk",
        "chunk_index": chunk["chunk_index"],
        "start_page": chunk["start_page"],
        "end_page": chunk["end_page"],
        "page_numbers": chunk["page_numbers"],
        "text": chunk["text"],
    })

    relationships.append({
        "source": doc_id,
        "source_label": "Document",
        "type": "HAS_CHUNK",
        "target": c_id,
        "target_label": "Chunk",
        "properties": {},
    })

    for page_str, page_text in chunk["pages"].items():
        page_num = int(page_str)
        p_id = f"page_{page_num:02d}"

        nodes["Page"].append({
            "node_id": p_id,
            "label": "Page",
            "page_number": page_num,
            "chunk_id": c_id,
            "has_text": bool(page_text.strip()),
            "text": page_text,
        })

        relationships.append({
            "source": c_id,
            "source_label": "Chunk",
            "type": "CONTAINS_PAGE",
            "target": p_id,
            "target_label": "Page",
            "properties": {"page_number": page_num},
        })

        # Extract year entities on this page
        mentions = extract_years_from_page(page_text)
        page_entity_groups = defaultdict(list)
        for item in mentions:
            e_id = item["entity_id"]
            if e_id not in entity_registry:
                entity_registry[e_id] = {
                    "node_id": e_id,
                    "label": "ENTITIES",
                    "entity_type": "Year",
                    "name": item["name"],
                    "year_value": item["year_value"],
                    "calendar": item["calendar"],
                    "total_mentions": 0,
                    "pages": set(),
                    "chunks": set(),
                }
            entity_registry[e_id]["total_mentions"] += 1
            entity_registry[e_id]["pages"].add(page_num)
            entity_registry[e_id]["chunks"].add(c_id)
            page_entity_groups[e_id].append(item["context"])

            entity_chunk_tracker[(e_id, c_id)]["count"] += 1
            entity_chunk_tracker[(e_id, c_id)]["pages"].add(page_num)

        # Create Page <-> ENTITIES edges
        for e_id, contexts in page_entity_groups.items():
            edge_props = {
                "mention_count": len(contexts),
                "contexts": contexts[:3],  # keep up to 3 snippets per page
            }
            relationships.append({
                "source": p_id,
                "source_label": "Page",
                "type": "MENTIONS_ENTITY",
                "target": e_id,
                "target_label": "ENTITIES",
                "properties": edge_props,
            })
            relationships.append({
                "source": e_id,
                "source_label": "ENTITIES",
                "type": "APPEARS_IN_PAGE",
                "target": p_id,
                "target_label": "Page",
                "properties": edge_props,
            })

# Finalize ENTITIES list and ENTITIES -> Chunk edges
for e_id, ent in sorted(entity_registry.items(), key=lambda x: (x[1]["calendar"], x[1]["year_value"])):
    ent["pages"] = sorted(ent["pages"])
    ent["chunks"] = sorted(ent["chunks"])
    nodes["ENTITIES"].append(ent)

for (e_id, c_id), stats in sorted(entity_chunk_tracker.items()):
    relationships.append({
        "source": e_id,
        "source_label": "ENTITIES",
        "type": "APPEARS_IN_CHUNK",
        "target": c_id,
        "target_label": "Chunk",
        "properties": {
            "mention_count": stats["count"],
            "pages": sorted(stats["pages"]),
        },
    })

graph_payload = {
    "schema": {
        "node_labels": ["Document", "Chunk", "Page", "ENTITIES"],
        "relationship_types": [
            "HAS_CHUNK",
            "CONTAINS_PAGE",
            "MENTIONS_ENTITY",
            "APPEARS_IN_PAGE",
            "APPEARS_IN_CHUNK",
        ],
    },
    "nodes": nodes,
    "relationships": relationships,
}

with open("step2_graph_data.json", "w", encoding="utf-8") as out_f:
    json.dump(graph_payload, out_f, indent=2, ensure_ascii=False)

# 4. Verification Summary
print("=== GRAPH NODE & RELATIONSHIP COUNTS ===")
for label, items in nodes.items():
    print(f"  :{label:<10} -> {len(items)} nodes")
print(f"  Total Edges -> {len(relationships)} relationships\n")

# Display extracted :ENTITIES table
entities_df = pd.DataFrame([
    {
        "Entity ID": e["node_id"],
        "Label": e["label"],
        "Type": e["entity_type"],
        "Year": e["name"],
        "Calendar": e["calendar"],
        "Mentions": e["total_mentions"],
        "Pages": str(e["pages"]),
        "Chunks": ", ".join(e["chunks"]),
    }
    for e in nodes["ENTITIES"]
])

print(f"=== EXTRACTED YEAR ENTITIES ({len(entities_df)} UNIQUE YEARS) ===")
display(entities_df)

=== GRAPH NODE & RELATIONSHIP COUNTS ===
  :Document   -> 1 nodes
  :Chunk      -> 14 nodes
  :Page       -> 70 nodes
  :ENTITIES   -> 42 nodes
  Total Edges -> 221 relationships

=== EXTRACTED YEAR ENTITIES (42 UNIQUE YEARS) ===


,Entity ID,Label,Type,Year,Calendar,Mentions,Pages,Chunks
0,year_AD_1333,ENTITIES,Year,1333 A.D.,AD,1,[57],chunk_12
1,year_AD_1347,ENTITIES,Year,1347 A.D.,AD,1,[11],chunk_03
2,year_AD_1426,ENTITIES,Year,1426 A.D.,AD,1,[21],chunk_05
3,year_AD_1429,ENTITIES,Year,1429 A.D.,AD,1,[11],chunk_03
4,year_AD_1432,ENTITIES,Year,1432 A.D.,AD,1,[12],chunk_03
5,year_AD_1435,ENTITIES,Year,1435 A.D.,AD,1,[12],chunk_03
6,year_AD_1459,ENTITIES,Year,1459 A.D.,AD,1,[12],chunk_03
7,year_AD_1461,ENTITIES,Year,1461 A.D.,AD,1,[13],chunk_03
8,year_AD_1463,ENTITIES,Year,1463 A.D.,AD,1,[14],chunk_03
9,year_AD_1469,ENTITIES,Year,1469 A.D.,AD,1,[14],chunk_03


In [5]:
pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 11.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 56.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyvis]


In [6]:
import json
from pyvis.network import Network
from IPython.display import HTML, display

# 1. Load the Step 2 graph data
with open("step2_graph_data.json", "r", encoding="utf-8") as f:
    graph_data = json.load(f)

nodes_dict = graph_data["nodes"]
relationships = graph_data["relationships"]

HIDE_BLANK_PAGES = True
SHOW_ENTITY_CHUNK_SHORTCUT = False

# 2. Initialize PyVis Network
net = Network(
    height="700px",
    width="100%",
    bgcolor="#0f172a",
    font_color="#f8fafc",
    directed=True,
    notebook=True,
    cdn_resources="in_line",
)

net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -65,
      "centralGravity": 0.012,
      "springLength": 115,
      "springConstant": 0.06,
      "damping": 0.45
    },
    "solver": "forceAtlas2Based",
    "stabilization": { "iterations": 180 }
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "navigationButtons": true
  }
}
""")

included_node_ids = set()

# 3. Add :Document Node (Gold Star)
for doc in nodes_dict["Document"]:
    nid = doc["node_id"]
    included_node_ids.add(nid)
    net.add_node(
        nid,
        label="Book: Antiquities of Bidar",
        title=(
            f"<b>Label:</b> :Document<br>"
            f"<b>File:</b> {doc['title']}<br>"
            f"<b>Pages:</b> {doc['total_pages']}"
        ),
        color="#f59e0b",
        shape="star",
        size=34,
        font={"size": 16, "color": "#fef3c7", "bold": True},
    )

# 4. Add :Chunk Nodes (Blue Boxes)
for chunk in nodes_dict["Chunk"]:
    nid = chunk["node_id"]
    included_node_ids.add(nid)
    net.add_node(
        nid,
        label=f"{nid}\n(pp.{chunk['start_page']}-{chunk['end_page']})",
        title=(
            f"<b>Label:</b> :Chunk<br>"
            f"<b>ID:</b> {nid}<br>"
            f"<b>Pages:</b> {chunk['start_page']} to {chunk['end_page']}"
        ),
        color="#3b82f6",
        shape="box",
        size=22,
        font={"size": 12, "color": "#ffffff"},
    )

# 5. Add :Page Nodes (Teal Dots)
active_pages = {
    rel["source"] for rel in relationships if rel["type"] == "MENTIONS_ENTITY"
}
for page in nodes_dict["Page"]:
    nid = page["node_id"]
    if HIDE_BLANK_PAGES and not page["has_text"] and nid not in active_pages:
        continue
    included_node_ids.add(nid)
    has_entities = nid in active_pages
    snippet = page["text"][:180].replace("\n", " ")
    net.add_node(
        nid,
        label=f"P.{page['page_number']}",
        title=(
            f"<b>Label:</b> :Page<br>"
            f"<b>Page Number:</b> {page['page_number']}<br>"
            f"<b>Parent Chunk:</b> {page['chunk_id']}<br>"
            f"<b>Snippet:</b> {snippet}..."
        ),
        color="#14b8a6" if has_entities else "#475569",
        shape="dot",
        size=15 if has_entities else 9,
        font={"size": 10, "color": "#cbd5e1"},
    )

# 6. Add :ENTITIES Nodes (Coral for A.D., Purple for A.H.)
for ent in nodes_dict["ENTITIES"]:
    nid = ent["node_id"]
    included_node_ids.add(nid)
    is_ah = ent["calendar"] == "AH"
    net.add_node(
        nid,
        label=ent["name"],
        title=(
            f"<b>Label:</b> :ENTITIES<br>"
            f"<b>Type:</b> {ent['entity_type']}<br>"
            f"<b>Value:</b> {ent['name']} ({ent['calendar']})<br>"
            f"<b>Mentions:</b> {ent['total_mentions']}<br>"
            f"<b>Pages:</b> {ent['pages']}<br>"
            f"<b>Chunks:</b> {', '.join(ent['chunks'])}"
        ),
        color="#c084fc" if is_ah else "#fb7185",
        shape="ellipse",
        size=18 + (ent["total_mentions"] * 3),
        font={"size": 12, "color": "#ffffff", "bold": True},
    )

# 7. Add Directed Relationships
EDGE_STYLES = {
    "HAS_CHUNK":        {"color": "#60a5fa", "label": "HAS_CHUNK",        "dashes": False, "width": 2.0},
    "CONTAINS_PAGE":    {"color": "#2dd4bf", "label": "CONTAINS_PAGE",    "dashes": False, "width": 1.2},
    "MENTIONS_ENTITY":  {"color": "#fda4af", "label": "MENTIONS_ENTITY",  "dashes": False, "width": 2.2},
    "APPEARS_IN_CHUNK": {"color": "#a78bfa", "label": "APPEARS_IN_CHUNK", "dashes": True,  "width": 1.0},
}

for rel in relationships:
    rel_type = rel["type"]
    if rel_type == "APPEARS_IN_PAGE":
        continue
    if rel_type == "APPEARS_IN_CHUNK" and not SHOW_ENTITY_CHUNK_SHORTCUT:
        continue

    src, tgt = rel["source"], rel["target"]
    if src not in included_node_ids or tgt not in included_node_ids:
        continue

    style = EDGE_STYLES[rel_type]
    props = rel.get("properties", {})
    tooltip = f"<b>Relationship:</b> [:{rel_type}]"
    if "contexts" in props:
        snippets = "<br>• ".join(props["contexts"])
        tooltip += f"<br><b>Mentions:</b> {props['mention_count']}<br><b>Context:</b><br>• {snippets}"

    net.add_edge(
        src,
        tgt,
        label=style["label"],
        title=tooltip,
        color=style["color"],
        dashes=style["dashes"],
        width=style["width"],
        font={"size": 8, "color": "#94a3b8", "strokeWidth": 0},
    )

# 8. Convert string titles to DOM HTMLElements + style dark-mode wrapped tooltips
html_str = net.generate_html()

tooltip_fix = """
<style>
  div.vis-tooltip {
    background-color: #1e293b !important;
    color: #f8fafc !important;
    border: 1px solid #475569 !important;
    border-radius: 8px !important;
    padding: 10px 14px !important;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif !important;
    font-size: 13px !important;
    line-height: 1.45 !important;
    max-width: 360px !important;
    white-space: normal !important;
    box-shadow: 0 10px 25px rgba(0, 0, 0, 0.5) !important;
  }
</style>
<script>
  (function() {
    function toHtmlElement(html) {
      const div = document.createElement("div");
      div.innerHTML = html;
      return div;
    }
    if (typeof nodes !== "undefined") {
      nodes.forEach(function(n) {
        if (typeof n.title === "string") {
          nodes.update({ id: n.id, title: toHtmlElement(n.title) });
        }
      });
    }
    if (typeof edges !== "undefined") {
      edges.forEach(function(e) {
        if (typeof e.title === "string") {
          edges.update({ id: e.id, title: toHtmlElement(e.title) });
        }
      });
    }
  })();
</script>
</body>
"""

html_str = html_str.replace("</body>", tooltip_fix)

output_html = "bidar_entities_graph.html"
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html_str)

display(HTML(html_str))

In [7]:
import os

# Automatically read credentials from the uploaded Neo4j text file
credentials_path = "/content/Neo4j-c7554cbf-Created-2026-11-16.txt"

if os.path.exists(credentials_path):
    with open(credentials_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    extracted_password = None
    extracted_username = None
    for line in lines:
        # Match lines starting with password or NEO4J_PASSWORD, splitting on ':' or '='
        if "password" in line.lower():
            delimiter = "=" if "=" in line else ":"
            parts = line.split(delimiter)
            if len(parts) > 1:
                extracted_password = parts[1].strip().strip('"').strip("'")
        # Match lines starting with username or NEO4J_USERNAME
        elif "username" in line.lower():
            delimiter = "=" if "=" in line else ":"
            parts = line.split(delimiter)
            if len(parts) > 1:
                extracted_username = parts[1].strip().strip('"').strip("'")

    if extracted_password:
        NEO4J_URI = "neo4j+s://c7554cbf.databases.neo4j.io"
        NEO4J_USERNAME = extracted_username if extracted_username else "neo4j"
        NEO4J_PASSWORD = extracted_password
        print("Successfully parsed credentials from the file!")
        print(f"URI: {NEO4J_URI}")
        print(f"Username: {NEO4J_USERNAME}")
        print("Credentials successfully loaded in memory.")
    else:
        # Fallback manual parse if format differed
        for line in lines:
            parts = line.strip().split()
            for part in parts:
                if len(part) == 43 and '-' in part:
                    NEO4J_PASSWORD = part
                    NEO4J_URI = "neo4j+s://c7554cbf.databases.neo4j.io"
                    NEO4J_USERNAME = "neo4j"
                    print("Parsed raw password token directly from file block.")
                    break
else:
    print(f"Could not find the credentials file at {credentials_path}. Please make sure it is uploaded.")

Could not find the credentials file at /content/Neo4j-c7554cbf-Created-2026-11-16.txt. Please make sure it is uploaded.


In [8]:
 # Override credentials manually in-memory (this has highest priority in cell Dwsuqjr8n9Ql)
NEO4J_URI = "neo4j+s://c7554cbf.databases.neo4j.io"
NEO4J_USERNAME = "c7554cbf"
NEO4J_PASSWORD = "P3r-aMBbHdt0zPrf10351DIgLdgki-yGI99JV3DabnY"  # <-- Replace with your real password if changed

print("Manual override variables configured. You can now re-run cell Dwsuqjr8n9Ql!")

Manual override variables configured. You can now re-run cell Dwsuqjr8n9Ql!


In [9]:
!pip install -q neo4j sentence-transformers

import json
import time
from collections import defaultdict
from getpass import getpass
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# 1. Resolve Credentials using variables parsed from the text file in memory
if 'NEO4J_PASSWORD' in globals() and NEO4J_PASSWORD:
    URI_TO_USE = NEO4J_URI
    # Use the username currently in memory, do not force/default to 'neo4j'
    USER_TO_USE = NEO4J_USERNAME if 'NEO4J_USERNAME' in globals() else "neo4j"
    PWD_TO_USE = NEO4J_PASSWORD
    print(f"Using credentials parsed from file. Connecting as user: {USER_TO_USE}")
else:
    # Fallback to secrets or manual prompt inputs if memory is empty
    def get_aura_credentials():
        try:
            from google.colab import userdata
            uri = userdata.get("NEO4J_URI")
            user = userdata.get("NEO4J_USERNAME") or "neo4j"
            pwd = userdata.get("NEO4J_PASSWORD")
            if uri and pwd:
                print("Loaded credentials from Colab Secrets.")
                return uri.strip(), user.strip(), pwd.strip()
        except Exception:
            pass

        # Manual fallback inputs
        uri = input("Enter NEO4J_URI (e.g., neo4j+s://c7554cbf.databases.neo4j.io): ").strip()
        user = input("Enter NEO4J_USERNAME: ").strip() or "neo4j"
        pwd = getpass("Enter NEO4J_PASSWORD: ").strip()
        return uri, user, pwd

    URI_TO_USE, USER_TO_USE, PWD_TO_USE = get_aura_credentials()

# Automatically correct common typos
if "databaneo4j.io" in URI_TO_USE:
    URI_TO_USE = URI_TO_USE.replace("databaneo4j.io", "databases.neo4j.io")
    print(f"Corrected URI typo to: {URI_TO_USE}")

# 2. Verify Connection to Neo4j Aura
driver = GraphDatabase.driver(URI_TO_USE, auth=(USER_TO_USE, PWD_TO_USE))
driver.verify_connectivity()
print("Connected to Neo4j Aura successfully!")

# 3. Load Step 2 Graph Data & Compute Dense Embeddings for :Chunk Nodes
with open("step2_graph_data.json", "r", encoding="utf-8") as f:
    graph_data = json.load(f)

nodes_dict = graph_data["nodes"]
relationships = graph_data["relationships"]

print("Generating 384-dim embeddings for 14 chunks using all-MiniLM-L6-v2...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [c["text"] for c in nodes_dict["Chunk"]]
chunk_vectors = embed_model.encode(chunk_texts, normalize_embeddings=True).tolist()

for chunk_node, vec in zip(nodes_dict["Chunk"], chunk_vectors):
    chunk_node["embedding"] = vec

# 4. Create Uniqueness Constraints & Ingest Nodes + Relationships
ALLOWED_LABELS = {"Document", "Chunk", "Page", "ENTITIES"}
ALLOWED_REL_TYPES = {
    "HAS_CHUNK",
    "CONTAINS_PAGE",
    "MENTIONS_ENTITY",
    "APPEARS_IN_PAGE",
    "APPEARS_IN_CHUNK",
}

with driver.session() as session:
    # Uniqueness constraints (auto-create B-tree lookup indexes)
    for label in ALLOWED_LABELS:
        session.run(
            f"CREATE CONSTRAINT {label.lower()}_id_unique IF NOT EXISTS "
            f"FOR (n:{label}) REQUIRE n.node_id IS UNIQUE"
        )

    # Upsert :Document
    session.run(
        """
        UNWIND $rows AS row
        MERGE (d:Document {node_id: row.node_id})
        SET d.title = row.title,
            d.total_pages = row.total_pages
        """,
        rows=nodes_dict["Document"],
    )

    # Upsert :Chunk (with vector embeddings)
    session.run(
        """
        UNWIND $rows AS row
        MERGE (c:Chunk {node_id: row.node_id})
        SET c.chunk_index = row.chunk_index,
            c.start_page = row.start_page,
            c.end_page = row.end_page,
            c.page_numbers = row.page_numbers,
            c.text = row.text,
            c.embedding = row.embedding
        """,
        rows=nodes_dict["Chunk"],
    )

    # Upsert :Page
    session.run(
        """
        UNWIND $rows AS row
        MERGE (p:Page {node_id: row.node_id})
        SET p.page_number = row.page_number,
            p.chunk_id = row.chunk_id,
            p.has_text = row.has_text,
            p.text = row.text
        """,
        rows=nodes_dict["Page"],
    )

    # Upsert :ENTITIES
    session.run(
        """
        UNWIND $rows AS row
        MERGE (e:ENTITIES {node_id: row.node_id})
        SET e.entity_type = row.entity_type,
            e.name = row.name,
            e.year_value = row.year_value,
            e.calendar = row.calendar,
            e.total_mentions = row.total_mentions,
            e.pages = row.pages,
            e.chunks = row.chunks
        """,
        rows=nodes_dict["ENTITIES"],
    )

    # Group relationships by (source_label, type, target_label) and batch-insert
    grouped_rels = defaultdict(list)
    for rel in relationships:
        key = (rel["source_label"], rel["type"], rel["target_label"])
        grouped_rels[key].append(rel)

    for (src_lbl, rel_type, tgt_lbl), rel_batch in grouped_rels.items():
        if src_lbl not in ALLOWED_LABELS or tgt_lbl not in ALLOWED_LABELS or rel_type not in ALLOWED_REL_TYPES:
            continue
        cypher = f"""
        UNWIND $rows AS row
        MATCH (src:{src_lbl} {{node_id: row.source}})
        MATCH (tgt:{tgt_lbl} {{node_id: row.target}})
        MERGE (src)-[r:{rel_type}]->(tgt)
        SET r += row.properties
        """
        session.run(cypher, rows=rel_batch)

    # 5. Create Native Neo4j Vector Index on :Chunk(embedding)
    session.run(
        """
        CREATE VECTOR INDEX chunk_embeddings IF NOT EXISTS
        FOR (c:Chunk)
        ON (c.embedding)
        OPTIONS {
          indexConfig: {
            `vector.dimensions`: 384,
            `vector.similarity_function`: 'cosine'
          }
        }
        """
    )

    # Wait until the vector index reaches ONLINE state
    for _ in range(15):
        res = session.run(
            "SHOW VECTOR INDEXES YIELD name, state WHERE name = 'chunk_embeddings' RETURN state"
        ).single()
        if res and res["state"] == "ONLINE":
            print("Vector index 'chunk_embeddings' is ONLINE!")
            break
        time.sleep(1)

print("Ingestion complete: All nodes, relationships, and embeddings are stored in Neo4j Aura.")

Using credentials parsed from file. Connecting as user: c7554cbf
Connected to Neo4j Aura successfully!
Generating 384-dim embeddings for 14 chunks using all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector index 'chunk_embeddings' is ONLINE!
Ingestion complete: All nodes, relationships, and embeddings are stored in Neo4j Aura.


In [10]:
import pandas as pd

def run_cypher_df(cypher_query: str, params: dict = None) -> pd.DataFrame:
    with driver.session() as session:
        result = session.run(cypher_query, params or {})
        return pd.DataFrame([record.data() for record in result])

# --- TEST 1: Verify Node & Relationship Counts in Neo4j Aura ---
print("=== TEST 1: NEO4J AURA NODE COUNTS ===")
df_counts = run_cypher_df("""
MATCH (n)
RETURN labels(n)[0] AS NodeLabel, count(n) AS TotalNodes
ORDER BY TotalNodes DESC
""")
display(df_counts)

# --- TEST 2: Multi-Page / Multi-Chunk Year Entities + Context ---
print("\n=== TEST 2: YEARS SPANNING MULTIPLE PAGES (GRAPH TRAVERSAL) ===")
df_bridges = run_cypher_df("""
MATCH (p:Page)-[r:MENTIONS_ENTITY]->(e:ENTITIES)
WHERE size(e.pages) > 1
RETURN
  e.name AS YearEntity,
  e.calendar AS Calendar,
  p.page_number AS Page,
  p.chunk_id AS Chunk,
  r.contexts[0] AS ExtractedSnippet
ORDER BY e.year_value ASC, p.page_number ASC
""")
display(df_bridges)

# --- TEST 3: Pages Linking Both Hijri (A.H.) and Common Era (A.D.) Years ---
print("\n=== TEST 3: DUAL-CALENDAR PAGES (A.H. + A.D. CO-OCCURRENCE) ===")
df_dual_cal = run_cypher_df("""
MATCH (p:Page)-[:MENTIONS_ENTITY]->(ah:ENTITIES {calendar: 'AH'}),
      (p)-[:MENTIONS_ENTITY]->(ad:ENTITIES {calendar: 'AD'})
RETURN
  p.page_number AS Page,
  p.chunk_id AS Chunk,
  collect(DISTINCT ah.name) AS Hijri_AH_Years,
  collect(DISTINCT ad.name) AS CommonEra_AD_Years
ORDER BY p.page_number ASC
""")
display(df_dual_cal)

# --- TEST 4: Hybrid Vector Search + Graph Entity Traversal ---
test_question = "When was the Madrasah of Mahmud Gawan built and when was it damaged?"
print(f"\n=== TEST 4: HYBRID VECTOR + GRAPH QUERY ===\nQuestion: '{test_question}'\n")

query_vec = embed_model.encode(test_question, normalize_embeddings=True).tolist()

df_hybrid = run_cypher_df(
    """
    CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_vector)
    YIELD node AS chunk, score
    OPTIONAL MATCH (chunk)-[:CONTAINS_PAGE]->(p:Page)-[:MENTIONS_ENTITY]->(e:ENTITIES)
    WITH chunk, score, p, e
    ORDER BY p.page_number ASC, e.calendar ASC, e.year_value ASC
    RETURN
      chunk.node_id AS ChunkID,
      toString(chunk.start_page) + '–' + toString(chunk.end_page) AS PageWindow,
      round(score, 4) AS VectorScore,
      collect(DISTINCT e.name + ' (p.' + toString(p.page_number) + ')') AS LinkedYearEntities,
      left(replace(chunk.text, '\\n', ' '), 180) + '...' AS ChunkPreview
    ORDER BY VectorScore DESC
    """,
    {"top_k": 3, "query_vector": query_vec},
)
display(df_hybrid)

=== TEST 1: NEO4J AURA NODE COUNTS ===


,NodeLabel,TotalNodes
0,Page,70
1,ENTITIES,42
2,Chunk,14
3,THEME,7
4,Document,1



=== TEST 2: YEARS SPANNING MULTIPLE PAGES (GRAPH TRAVERSAL) ===


,YearEntity,Calendar,Page,Chunk,ExtractedSnippet
0,1579 A.D.,AD,20,chunk_04,Barid had assumed regalia he did not take the ...
1,1579 A.D.,AD,33,chunk_07,f his death is contained in-' Found rest in he...
2,1656 A.D.,AD,20,chunk_04,Bijapur. Later Barid Shahi kings. Bidar an- ne...
3,1656 A.D.,AD,34,chunk_07,ient times and worshipped an image of the lion...
4,1686 A.D.,AD,10,chunk_02,lasts till it was finally captured by the host...
5,1686 A.D.,AD,26,chunk_06,t Prince Aurangzeb jubilant at his conquest of...
6,1992 A.D.,AD,7,chunk_02,F BIDAR G. YAZDANI S ASIAN EDUCATIONAL SERVICE...
7,1992 A.D.,AD,8,chunk_02,"EET, MADRAS-600014. AES a 285- First Published..."



=== TEST 3: DUAL-CALENDAR PAGES (A.H. + A.D. CO-OCCURRENCE) ===


,Page,Chunk,Hijri_AH_Years,CommonEra_AD_Years
0,12,chunk_03,[836 A.H.],"[1432 A.D., 1435 A.D., 1459 A.D.]"
1,16,chunk_04,[887 A.H.],[1482 A.D.]
2,18,chunk_04,[896 A.H.],[1490 A.D.]
3,21,chunk_05,[830 A.H.],"[1426 A.D., 1667 A.D.]"
4,28,chunk_06,[877 A.H.],[1472 A.D.]
5,33,chunk_07,"[984 A.H., 987 A.H.]","[1576 A.D., 1579 A.D.]"



=== TEST 4: HYBRID VECTOR + GRAPH QUERY ===
Question: 'When was the Madrasah of Mahmud Gawan built and when was it damaged?'



,ChunkID,PageWindow,VectorScore,LinkedYearEntities,ChunkPreview
0,chunk_04,16–20,0.7304,"[1482 A.D. (p.16), 887 A.H. (p.16), 1490 A.D. ...",--- [Page 16] --- Jilas who jugu 3. TO Jail s'...
1,chunk_08,36–40,0.7135,[],"--- [Page 36] --- BIBLIOGRAPHY. (I) Bakhshi, N..."
2,chunk_13,61–65,0.7104,[],--- [Page 61] --- IIIX PLATE 000 NORTHERN END ...


In [11]:
import time
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# Re-use driver & embed_model from Step 3 (or initialize if kernel restarted)
if "embed_model" not in globals():
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 1. Fetch all active :Page nodes from Neo4j Aura
with driver.session() as session:
    records = session.run("""
        MATCH (p:Page)
        WHERE p.has_text = true AND size(trim(p.text)) > 0
        RETURN p.node_id AS node_id, p.page_number AS page_number, p.text AS text
        ORDER BY p.page_number ASC
    """).data()

print(f"Encoding {len(records)} active :Page nodes into 384-dim vectors...")

page_texts = [f"Page {r['page_number']}: {r['text']}" for r in records]
page_vectors = embed_model.encode(page_texts, normalize_embeddings=True).tolist()

rows_to_update = [
    {"node_id": r["node_id"], "embedding": vec}
    for r, vec in zip(records, page_vectors)
]

# 2. Store embeddings on :Page nodes & create 'page_embeddings' vector index
with driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MATCH (p:Page {node_id: row.node_id})
        SET p.embedding = row.embedding
    """, rows=rows_to_update)

    session.run("""
        CREATE VECTOR INDEX page_embeddings IF NOT EXISTS
        FOR (p:Page)
        ON (p.embedding)
        OPTIONS {
          indexConfig: {
            `vector.dimensions`: 384,
            `vector.similarity_function`: 'cosine'
          }
        }
    """)

    for _ in range(15):
        res = session.run(
            "SHOW VECTOR INDEXES YIELD name, state WHERE name = 'page_embeddings' RETURN state"
        ).single()
        if res and res["state"] == "ONLINE":
            print("Vector index 'page_embeddings' is ONLINE!")
            break
        time.sleep(1)

Encoding 49 active :Page nodes into 384-dim vectors...
Vector index 'page_embeddings' is ONLINE!


In [13]:
!pip install -q boto3 neo4j sentence-transformers

import re
import boto3
from getpass import getpass
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from IPython.display import Markdown, display

# ==============================================================================
# 1. CONNECT TO NEO4J AURA (Enforced verified overrides to bypass stale globals)
# ==============================================================================
URI_TO_USE = "neo4j+s://c7554cbf.databases.neo4j.io"
USER_TO_USE = "c7554cbf"
PWD_TO_USE = "P3r-aMBbHdt0zPrf10351DIgLdgki-yGI99JV3DabnY"

driver = GraphDatabase.driver(URI_TO_USE, auth=(USER_TO_USE, PWD_TO_USE))
driver.verify_connectivity()
print(f"Connected to Neo4j Aura successfully as user: {USER_TO_USE}!")

# ==============================================================================
# 2. CONNECT TO AMAZON BEDROCK RUNTIME
# ==============================================================================
def get_secret_or_prompt(key_name: str, prompt_text: str, is_secret: bool = True) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(key_name)
        if val:
            return val.strip()
    except Exception:
        pass
    return (getpass(prompt_text) if is_secret else input(prompt_text)).strip()

AWS_REGION = input("Enter AWS Region [default: us-east-1]: ").strip() or "us-east-1"
AWS_ACCESS_KEY_ID = get_secret_or_prompt("AWS_ACCESS_KEY_ID", "Enter AWS_ACCESS_KEY_ID: ")
AWS_SECRET_ACCESS_KEY = get_secret_or_prompt("AWS_SECRET_ACCESS_KEY", "Enter AWS_SECRET_ACCESS_KEY: ")

# Choose your Bedrock Foundation Model ID:
#   - "us.amazon.nova-2-lite-v1:0"          (Amazon Nova 2 Lite - US cross-region profile)
#   - "us.meta.llama3-1-8b-instruct-v1:0"   (Meta Llama 3.1 8B Instruct - US cross-region profile)
#   - "meta.llama3-1-8b-instruct-v1:0"      (Direct regional model ID, e.g., us-east-1 / us-west-2)
BEDROCK_MODEL_ID = "us.amazon.nova-2-lite-v1:0"

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
print(f"Configured Amazon Bedrock client ({AWS_REGION}) with model: {BEDROCK_MODEL_ID}")

# ==============================================================================
# 3. HYBRID GRAPHRAG RETRIEVER (ENTITY MATCH + PAGE VECTOR + 2-HOP EXPANSION)
# ==============================================================================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
YEAR_QUERY_RE = re.compile(r"\b(\d{3,4})\b")

def retrieve_graph_context(question: str, top_k_pages: int = 4) -> dict:
    query_vec = embed_model.encode(question, normalize_embeddings=True).tolist()
    candidate_years = [int(y) for y in YEAR_QUERY_RE.findall(question)]

    with driver.session() as session:
        # Path A: Direct :ENTITIES year lookup
        entity_hits = []
        if candidate_years:
            entity_hits = session.run(
                """
                MATCH (e:ENTITIES)<-[:MENTIONS_ENTITY]-(p:Page)<-[:CONTAINS_PAGE]-(c:Chunk)
                WHERE e.year_value IN $years
                RETURN DISTINCT
                  p.node_id AS page_id,
                  p.page_number AS page_number,
                  c.node_id AS chunk_id,
                  1.0 AS score,
                  'Entity Match (' + e.name + ')' AS retrieval_source
                """,
                years=candidate_years,
            ).data()

        # Path B: Page-level vector similarity search
        vector_hits = session.run(
            """
            CALL db.index.vector.queryNodes('page_embeddings', $top_k, $query_vector)
            YIELD node AS p, score
            MATCH (c:Chunk)-[:CONTAINS_PAGE]->(p)
            RETURN
              p.node_id AS page_id,
              p.page_number AS page_number,
              c.node_id AS chunk_id,
              round(score, 4) AS score,
              'Vector Search' AS retrieval_source
            """,
            top_k=top_k_pages,
            query_vector=query_vec,
        ).data()

        # Deduplicate seed pages
        seen_pages = {}
        for hit in entity_hits + vector_hits:
            pid = hit["page_id"]
            if pid not in seen_pages:
                seen_pages[pid] = hit
            else:
                seen_pages[pid]["retrieval_source"] += f" + {hit['retrieval_source']}"

        seed_page_ids = list(seen_pages.keys())[:5]

        # Path C: 2-Hop Graph Expansion (linked years + neighboring pages in same chunk)
        expanded_records = session.run(
            """
            MATCH (c:Chunk)-[:CONTAINS_PAGE]->(p:Page)
            WHERE p.node_id IN $seed_ids
            OPTIONAL MATCH (p)-[:MENTIONS_ENTITY]->(e:ENTITIES)
            OPTIONAL MATCH (c)-[:CONTAINS_PAGE]->(neighbor:Page)
            WHERE abs(neighbor.page_number - p.page_number) <= 2 AND neighbor.has_text = true
            WITH c, p,
                 collect(DISTINCT e.name + ' [' + e.calendar + ']') AS page_years,
                 collect(DISTINCT {page: neighbor.page_number, text: neighbor.text}) AS nearby_pages
            RETURN
              p.page_number AS page_number,
              c.node_id AS chunk_id,
              toString(c.start_page) + '-' + toString(c.end_page) AS chunk_window,
              p.text AS page_text,
              page_years,
              nearby_pages
            ORDER BY p.page_number ASC
            """,
            seed_ids=seed_page_ids,
        ).data()

    return {
        "seed_hits": list(seen_pages.values()),
        "expanded_pages": expanded_records,
    }

# ==============================================================================
# 4. SYNTHESIZE GROUNDED ANSWERS WITH AMAZON BEDROCK CONVERSE API
# ==============================================================================
def ask_bidar_bedrock(question: str, model_id: str = BEDROCK_MODEL_ID):
    retrieved = retrieve_graph_context(question)

    context_blocks = []
    included_page_nums = set()

    for item in retrieved["expanded_pages"]:
        p_num = item["page_number"]
        included_page_nums.add(p_num)
        years_str = ", ".join(item["page_years"]) if item["page_years"] else "None"
        context_blocks.append(
            f"=== [PRIMARY MATCH: Page {p_num} | {item['chunk_id']} (pp.{item['chunk_window']}) | "
            f"Graph Year Entities: {years_str}] ===\n{item['page_text']}"
        )

    for item in retrieved["expanded_pages"]:
        for nb in item["nearby_pages"]:
            nb_num = nb["page"]
            if nb_num not in included_page_nums and nb["text"].strip():
                included_page_nums.add(nb_num)
                context_blocks.append(
                    f"=== [2-HOP NEIGHBOR PAGE: Page {nb_num} | {item['chunk_id']}] ===\n{nb['text']}"
                )

    formatted_context = "\n\n".join(context_blocks)

    system_prompts = [{
        "text": (
            "You are an expert architectural historian analyzing 'The Antiquities of Bidar' (1917) by G. Yazdani. "
            "Answer the user's question strictly using ONLY the provided Neo4j GraphRAG context.\n"
            "Rules:\n"
            "1. Cite exact page numbers, chunk IDs, and year entities (both A.D. and A.H. when present) "
            "in brackets like [Page 28 | chunk_06 | 1472 A.D. / 877 A.H.].\n"
            "2. Explain historical details clearly and note minor OCR spelling quirks if obvious.\n"
            "3. If the context does not contain the answer, state that clearly without guessing."
        )
    }]

    messages = [{
        "role": "user",
        "content": [{
            "text": f"GRAPH RETRIEVED CONTEXT:\n{formatted_context}\n\nUSER QUESTION:\n{question}"
        }],
    }]

    response = bedrock_client.converse(
        modelId=model_id,
        messages=messages,
        system=system_prompts,
        inferenceConfig={
            "temperature": 0.2,
            "maxTokens": 1024,
        },
    )

    answer_text = response["output"]["message"]["content"][0]["text"]
    usage = response.get("usage", {})

    print("=" * 80)
    print(f"QUESTION: {question}")
    print("-" * 80)
    print("RETRIEVED GRAPH SEEDS:")
    for hit in retrieved["seed_hits"]:
        print(f"  • Page {hit['page_number']:02d} ({hit['chunk_id']}) | Score: {hit['score']} | Via: {hit['retrieval_source']}")
    print(f"BEDROCK TOKENS: Input={usage.get('inputTokens', 'N/A')} | Output={usage.get('outputTokens', 'N/A')}")
    print("=" * 80)
    display(Markdown(answer_text))

# ==============================================================================
# 5. RUN THE 3 TEST QUERIES
# ==============================================================================
test_questions = [
    "When was the Madrasah of Mahmud Gawan built and when was it damaged?",
    "What happened in Bidar in 1686 A.D. and which monuments or events are connected to that year?",
    "Compare the founding and completion dates of Bidar Fort in both Hijri (A.H.) and Common Era (A.D.) calendars.",
]

for q in test_questions:
    ask_bidar_bedrock(q)

Connected to Neo4j Aura successfully as user: c7554cbf!
Enter AWS Region [default: us-east-1]: us-east-1
Enter AWS_ACCESS_KEY_ID: ··········
Enter AWS_SECRET_ACCESS_KEY: ··········
Configured Amazon Bedrock client (us-east-1) with model: us.amazon.nova-2-lite-v1:0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

QUESTION: When was the Madrasah of Mahmud Gawan built and when was it damaged?
--------------------------------------------------------------------------------
RETRIEVED GRAPH SEEDS:
  • Page 15 (chunk_03) | Score: 0.7842 | Via: Vector Search
  • Page 35 (chunk_07) | Score: 0.7508 | Via: Vector Search
  • Page 30 (chunk_06) | Score: 0.7467 | Via: Vector Search
  • Page 61 (chunk_13) | Score: 0.7442 | Via: Vector Search
BEDROCK TOKENS: Input=4424 | Output=81


The Madrasah of Mahmud Gawan was built in **877 A.H. (1472 A.D.)**, as indicated in the text [Page 28 | chunk_06]. It suffered significant damage in **1107 A.H. (1696 A.D.)** due to a lightning strike, which destroyed half of its front and southern wing [Page 30 | chunk_06].

QUESTION: What happened in Bidar in 1686 A.D. and which monuments or events are connected to that year?
--------------------------------------------------------------------------------
RETRIEVED GRAPH SEEDS:
  • Page 10 (chunk_02) | Score: 1.0 | Via: Entity Match (1686 A.D.) + Vector Search
  • Page 26 (chunk_06) | Score: 1.0 | Via: Entity Match (1686 A.D.)
  • Page 05 (chunk_01) | Score: 0.8317 | Via: Vector Search
  • Page 01 (chunk_01) | Score: 0.7362 | Via: Vector Search
  • Page 55 (chunk_11) | Score: 0.7197 | Via: Vector Search
BEDROCK TOKENS: Input=3090 | Output=262


In 1686 A.D., Aurangzeb conquered Bidar, marking the end of its rule under the Barid Shahi dynasty and incorporating it into the Mughal Empire [Page 10 | chunk_02 (pp.6-10) | Graph Year Entities: 1686 A.D. [AD]].

The conquest is directly connected to several key events and monuments:

1. **Khutba in the Name of the Emperor**: Following his conquest, Aurangzeb hastened to the mosque in Bidar to have the Khutba (Friday sermon) read in the name of the Emperor (Shahjahan) [Page 26 | chunk_06 (pp.26-30) | Graph Year Entities: 1686 A.D. [AD]]. This act symbolized the formal transfer of religious and political authority to Mughal rule.

2. **Capture of Bidar**: The conquest itself represents the final military event in Bidar's history under Islamic dynasties, concluding its period as a center of Bahmani and Barid Shahi power [Page 10 | chunk_02 (pp.6-10) | Graph Year Entities: 1686 A.D. [AD]].

These events collectively signify the culmination of Bidar's historical trajectory under Islamic rule and its integration into the Mughal Empire.

QUESTION: Compare the founding and completion dates of Bidar Fort in both Hijri (A.H.) and Common Era (A.D.) calendars.
--------------------------------------------------------------------------------
RETRIEVED GRAPH SEEDS:
  • Page 05 (chunk_01) | Score: 0.7318 | Via: Vector Search
  • Page 21 (chunk_05) | Score: 0.7313 | Via: Vector Search
  • Page 10 (chunk_02) | Score: 0.7216 | Via: Vector Search
  • Page 55 (chunk_11) | Score: 0.721 | Via: Vector Search
BEDROCK TOKENS: Input=2895 | Output=144


The Bidar Fort was founded and completed during the reign of Ahmad Shah Wali Bahmani. According to the text:

- **Founded/Completed in Hijri Calendar**: 830 A.H.  
- **Founded/Completed in Common Era (A.D.)**: 1426 A.D.  

This information is explicitly stated on **Page 21 | chunk_05**, which notes: *"It was begun, as already stated, in 830 A.H. (1426 A.D.) during the reign of Ahmad Shãh Wali, and took six years to complete."*  

Thus, the fort's construction began and was completed in **830 A.H. / 1426 A.D.**.

In [14]:
!pip install -q pyvis

import re
from pyvis.network import Network
from IPython.display import HTML, Markdown, display

YEAR_QUERY_RE = re.compile(r"\b(\d{3,4})\b")

def retrieve_graph_with_subgraph(question: str, top_k_pages: int = 4) -> dict:
    query_vec = embed_model.encode(question, normalize_embeddings=True).tolist()
    candidate_years = [int(y) for y in YEAR_QUERY_RE.findall(question)]

    with driver.session() as session:
        # Path A: Direct :ENTITIES year match
        entity_hits = []
        if candidate_years:
            entity_hits = session.run(
                """
                MATCH (e:ENTITIES)<-[:MENTIONS_ENTITY]-(p:Page)<-[:CONTAINS_PAGE]-(c:Chunk)
                WHERE e.year_value IN $years
                RETURN DISTINCT
                  p.node_id AS page_id,
                  p.page_number AS page_number,
                  c.node_id AS chunk_id,
                  1.0 AS score,
                  'Entity Match (' + e.name + ')' AS retrieval_source
                """,
                years=candidate_years,
            ).data()

        # Path B: Cypher 25 native vector SEARCH clause (no deprecation warning)
        vector_hits = session.run(
            """
            CYPHER 25
            MATCH (p:Page)
            SEARCH p IN (VECTOR INDEX page_embeddings FOR $query_vector LIMIT $top_k) SCORE AS score
            MATCH (c:Chunk)-[:CONTAINS_PAGE]->(p)
            RETURN
              p.node_id AS page_id,
              p.page_number AS page_number,
              c.node_id AS chunk_id,
              round(score, 4) AS score,
              'Vector Search' AS retrieval_source
            """,
            top_k=top_k_pages,
            query_vector=query_vec,
        ).data()

        # Deduplicate seed pages
        seen_pages = {}
        for hit in entity_hits + vector_hits:
            pid = hit["page_id"]
            if pid not in seen_pages:
                seen_pages[pid] = hit
            else:
                seen_pages[pid]["retrieval_source"] += f" + {hit['retrieval_source']}"

        seed_page_ids = list(seen_pages.keys())[:5]

        # Path C: 2-Hop expansion + collect subgraph nodes & edges for visualization
        expanded_records = session.run(
            """
            MATCH (c:Chunk)-[:CONTAINS_PAGE]->(p:Page)
            WHERE p.node_id IN $seed_ids
            OPTIONAL MATCH (p)-[r1:MENTIONS_ENTITY]->(e:ENTITIES)
            OPTIONAL MATCH (c)-[:CONTAINS_PAGE]->(neighbor:Page)
            WHERE abs(neighbor.page_number - p.page_number) <= 2 AND neighbor.has_text = true
            OPTIONAL MATCH (neighbor)-[r2:MENTIONS_ENTITY]->(ne:ENTITIES)
            WITH c, p,
                 collect(DISTINCT {
                   id: e.node_id, name: e.name, calendar: e.calendar,
                   snippet: r1.contexts[0]
                 }) AS page_entities,
                 collect(DISTINCT {
                   id: neighbor.node_id, page: neighbor.page_number, text: neighbor.text
                 }) AS nearby_pages,
                 collect(DISTINCT {
                   page_id: neighbor.node_id, id: ne.node_id, name: ne.name,
                   calendar: ne.calendar, snippet: r2.contexts[0]
                 }) AS neighbor_entities
            RETURN
              p.node_id AS page_id,
              p.page_number AS page_number,
              c.node_id AS chunk_id,
              toString(c.start_page) + '-' + toString(c.end_page) AS chunk_window,
              p.text AS page_text,
              page_entities,
              nearby_pages,
              neighbor_entities
            ORDER BY p.page_number ASC
            """,
            seed_ids=seed_page_ids,
        ).data()

    return {
        "seed_hits": list(seen_pages.values()),
        "expanded_pages": expanded_records,
    }


def render_evidence_subgraph(question: str, retrieved: dict, html_file: str = "query_subgraph.html"):
    net = Network(
        height="520px",
        width="100%",
        bgcolor="#0f172a",
        font_color="#f8fafc",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )

    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -75,
          "centralGravity": 0.015,
          "springLength": 120,
          "springConstant": 0.07,
          "damping": 0.45
        },
        "solver": "forceAtlas2Based",
        "stabilization": { "iterations": 150 }
      },
      "interaction": { "hover": true, "tooltipDelay": 80, "navigationButtons": true }
    }
    """)

    seed_lookup = {h["page_id"]: h for h in retrieved["seed_hits"]}
    added_nodes = set()
    added_edges = set()

    # Central Query Node
    short_q = question if len(question) <= 45 else question[:42] + "..."
    net.add_node(
        "USER_QUERY",
        label=f"Query:\n{short_q}",
        title=f"<b>User Question:</b><br>{question}",
        color="#f59e0b",
        shape="star",
        size=28,
        font={"size": 13, "color": "#fef3c7", "bold": True},
    )
    added_nodes.add("USER_QUERY")

    for item in retrieved["expanded_pages"]:
        c_id = item["chunk_id"]
        p_id = item["page_id"]
        p_num = item["page_number"]

        # 1. Chunk Node
        if c_id not in added_nodes:
            net.add_node(
                c_id,
                label=f"{c_id}\n(pp.{item['chunk_window']})",
                title=f"<b>Label:</b> :Chunk<br><b>ID:</b> {c_id}<br><b>Window:</b> Pages {item['chunk_window']}",
                color="#3b82f6",
                shape="box",
                size=22,
                font={"size": 12, "color": "#ffffff"},
            )
            added_nodes.add(c_id)

        # 2. Primary Seed Page Node
        if p_id not in added_nodes:
            hit_info = seed_lookup.get(p_id, {})
            snippet = item["page_text"][:180].replace("\n", " ")
            net.add_node(
                p_id,
                label=f"Page {p_num}\n(Seed)",
                title=(
                    f"<b>Label:</b> :Page (Primary Seed)<br>"
                    f"<b>Matched Via:</b> {hit_info.get('retrieval_source', 'Graph')}<br>"
                    f"<b>Score:</b> {hit_info.get('score', 'N/A')}<br>"
                    f"<b>Excerpt:</b> {snippet}..."
                ),
                color="#10b981",
                shape="dot",
                size=19,
                font={"size": 12, "color": "#ecfdf5", "bold": True},
            )
            added_nodes.add(p_id)

        if ("USER_QUERY", p_id) not in added_edges:
            net.add_edge("USER_QUERY", p_id, label="MATCHED", color="#fbbf24", dashes=True, width=1.8)
            added_edges.add(("USER_QUERY", p_id))

        if (c_id, p_id) not in added_edges:
            net.add_edge(c_id, p_id, label="CONTAINS_PAGE", color="#60a5fa", width=1.5)
            added_edges.add((c_id, p_id))

        # 3. 2-Hop Neighbor Pages in same Chunk
        for nb in item["nearby_pages"]:
            nb_id = nb["id"]
            if not nb_id or not nb["text"].strip():
                continue
            if nb_id not in added_nodes:
                snippet = nb["text"][:180].replace("\n", " ")
                net.add_node(
                    nb_id,
                    label=f"Page {nb['page']}\n(2-Hop)",
                    title=(
                        f"<b>Label:</b> :Page (2-Hop Context Neighbor)<br>"
                        f"<b>Parent Chunk:</b> {c_id}<br>"
                        f"<b>Excerpt:</b> {snippet}..."
                    ),
                    color="#14b8a6",
                    shape="dot",
                    size=14,
                    font={"size": 10, "color": "#cbd5e1"},
                )
                added_nodes.add(nb_id)
            if (c_id, nb_id) not in added_edges:
                net.add_edge(c_id, nb_id, label="CONTAINS_PAGE", color="#2dd4bf", width=1.1)
                added_edges.add((c_id, nb_id))

        # 4. Connected :ENTITIES (Years)
        all_ents = [(p_id, e) for e in item["page_entities"] if e["id"]] + [
            (ne["page_id"], ne) for ne in item["neighbor_entities"] if ne["id"] and ne["page_id"] in added_nodes
        ]
        for src_page_id, ent in all_ents:
            e_id = ent["id"]
            if e_id not in added_nodes:
                is_ah = ent["calendar"] == "AH"
                net.add_node(
                    e_id,
                    label=ent["name"],
                    title=(
                        f"<b>Label:</b> :ENTITIES (Year)<br>"
                        f"<b>Value:</b> {ent['name']}<br>"
                        f"<b>Calendar:</b> {ent['calendar']}"
                    ),
                    color="#c084fc" if is_ah else "#fb7185",
                    shape="ellipse",
                    size=18,
                    font={"size": 12, "color": "#ffffff", "bold": True},
                )
                added_nodes.add(e_id)
            if (src_page_id, e_id) not in added_edges:
                ctx = ent.get("snippet") or ""
                net.add_edge(
                    src_page_id,
                    e_id,
                    label="MENTIONS_ENTITY",
                    title=f"<b>[:MENTIONS_ENTITY]</b><br>{ctx}",
                    color="#fda4af",
                    width=1.8,
                )
                added_edges.add((src_page_id, e_id))

    html_str = net.generate_html()
    tooltip_fix = """
    <style>
      div.vis-tooltip {
        background-color: #1e293b !important; color: #f8fafc !important;
        border: 1px solid #475569 !important; border-radius: 8px !important;
        padding: 10px 14px !important; font-family: sans-serif !important;
        font-size: 13px !important; line-height: 1.45 !important;
        max-width: 360px !important; white-space: normal !important;
      }
    </style>
    <script>
      (function() {
        function toEl(h) { const d = document.createElement("div"); d.innerHTML = h; return d; }
        if (typeof nodes !== "undefined") nodes.forEach(n => { if (typeof n.title === "string") nodes.update({id: n.id, title: toEl(n.title)}); });
        if (typeof edges !== "undefined") edges.forEach(e => { if (typeof e.title === "string") edges.update({id: e.id, title: toEl(e.title)}); });
      })();
    </script></body>
    """
    html_str = html_str.replace("</body>", tooltip_fix)
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html_str)
    display(HTML(html_str))


def ask_and_visualize(question: str, model_id: str = BEDROCK_MODEL_ID):
    retrieved = retrieve_graph_with_subgraph(question)

    context_blocks = []
    included_page_nums = set()

    for item in retrieved["expanded_pages"]:
        p_num = item["page_number"]
        included_page_nums.add(p_num)
        years_str = ", ".join(
            sorted({f"{e['name']} [{e['calendar']}]" for e in item["page_entities"] if e["id"]})
        ) or "None"
        context_blocks.append(
            f"=== [PRIMARY MATCH: Page {p_num} | {item['chunk_id']} (pp.{item['chunk_window']}) | "
            f"Graph Year Entities: {years_str}] ===\n{item['page_text']}"
        )

    for item in retrieved["expanded_pages"]:
        for nb in item["nearby_pages"]:
            nb_num = nb["page"]
            if nb_num not in included_page_nums and nb["text"].strip():
                included_page_nums.add(nb_num)
                context_blocks.append(
                    f"=== [2-HOP NEIGHBOR PAGE: Page {nb_num} | {item['chunk_id']}] ===\n{nb['text']}"
                )

    formatted_context = "\n\n".join(context_blocks)

    system_prompts = [{
        "text": (
            "You are an expert architectural historian analyzing 'The Antiquities of Bidar' (1917) by G. Yazdani. "
            "Answer strictly using ONLY the provided Neo4j GraphRAG context. "
            "Cite exact page numbers, chunk IDs, and year entities (both A.D. and A.H. when present) "
            "in brackets like [Page 28 | chunk_06 | 1472 A.D. / 877 A.H.]."
        )
    }]

    response = bedrock_client.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": f"CONTEXT:\n{formatted_context}\n\nQUESTION:\n{question}"}]}],
        system=system_prompts,
        inferenceConfig={"temperature": 0.2, "maxTokens": 1024},
    )

    print("=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)
    display(Markdown(response["output"]["message"]["content"][0]["text"]))
    print("\nSUBGRAPH EVIDENCE MAP (★ Query | ■ Chunk | ● Green=Seed Page | ● Teal=2-Hop Page | ⬭ Coral=A.D. | ⬭ Purple=A.H.):")
    render_evidence_subgraph(question, retrieved)

# Test end-to-end GraphRAG + Visual Evidence Subgraph on a new question!
ask_and_visualize("Tell me about the Tomb of Ali Barid, including its dates in A.H. and A.D. and architectural features.")

QUESTION: Tell me about the Tomb of Ali Barid, including its dates in A.H. and A.D. and architectural features.


The Tomb of 'Ali Barid, the third ruler of the Barid Shahi line, is an imposing structure situated on an elevated plain to the west of Bidar [Page 31 | chunk_07]. Originally surrounded by elegant gardens with water channels and fountains, the tomb now stands amidst ruins. Its architectural features include:

- **Plan and Structure**: The tomb is square in plan (57 feet each way) with an open arch on all four sides. It is crowned by a turnip-shaped dome terminating in a spire, with a total height of 106 feet from the ground [Page 31 | chunk_07].
  
- **Interior**: Unlike other Muhammadan tombs that are typically dark and somber, the interior of 'Ali Barid's tomb is well-lit due to large side openings that admit plenty of light and air [Page 31 | chunk_07]. The interior is highly ornamented, featuring several courses of exquisite enamel-work arranged in calligraphical devices along the walls [Page 31 | chunk_07].

- **Construction Materials**: The tomb is built of hard black trap up to the drum of the dome, which is mounted with a brick and lime top, likely to protect the building from extra weight [Page 31 | chunk_07].

- **Inscriptions**: The tomb contains elegant inscriptions composed in a refined style. One notable inscription includes a hemistich that provides the date of the building's construction: "Style the dome the Vault of Divine light (984 A.H.= 1576 A.D.)" [Page 33 | chunk_07].

- **Date of Death**: The date of 'Ali Barid's death is contained in another inscription: "Found rest in heaven" which corresponds to 987 A.H. (1579 A.D.) [Page 33 | chunk_07].

In summary, the Tomb of 'Ali Barid is distinguished by its perfect proportions, well-lit interior, and intricate enamel-work, and it stands as a significant monument from the Barid Shahi period, with construction dated to 984 A.H. (1576 A.D.) and the ruler's death in 987 A.H. (1579 A.D.).


SUBGRAPH EVIDENCE MAP (★ Query | ■ Chunk | ● Green=Seed Page | ● Teal=2-Hop Page | ⬭ Coral=A.D. | ⬭ Purple=A.H.):


In [15]:
!pip install -q pyvis ipywidgets neo4j sentence-transformers boto3

import re
import time
import boto3
import ipywidgets as widgets
from pyvis.network import Network
from IPython.display import HTML, Markdown, display

# ==============================================================================
# 1. REUSE OR INITIALIZE CONNECTIONS (NEO4J AURA + EMBEDDINGS + AMAZON BEDROCK)
# ==============================================================================
# Automatically bind pre-existing connections from preceding steps if available
URI_TO_USE = globals().get('NEO4J_URI', 'neo4j+s://c7554cbf.databases.neo4j.io')
USER_TO_USE = globals().get('USER_TO_USE') or globals().get('NEO4J_USERNAME', 'c7554cbf')
PWD_TO_USE = globals().get('PWD_TO_USE') or globals().get('NEO4J_PASSWORD')

if 'driver' not in globals():
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(URI_TO_USE, auth=(USER_TO_USE, PWD_TO_USE))
    driver.verify_connectivity()
    print("Connected to Neo4j Aura successfully!")
else:
    driver = globals()['driver']
    print("Re-using existing Neo4j connection.")

if 'embed_model' not in globals():
    from sentence_transformers import SentenceTransformer
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("Loaded sentence embedding model.")
else:
    embed_model = globals()['embed_model']
    print("Re-using loaded sentence embedding model.")

if 'bedrock_client' not in globals():
    # Fallback init if credentials are in global variables but client was not initialized
    AWS_REGION = globals().get('AWS_REGION', 'us-east-1')
    AWS_ACCESS_KEY_ID = globals().get('AWS_ACCESS_KEY_ID')
    AWS_SECRET_ACCESS_KEY = globals().get('AWS_SECRET_ACCESS_KEY')
    BEDROCK_MODEL_ID = globals().get('BEDROCK_MODEL_ID', 'us.amazon.nova-2-lite-v1:0')

    bedrock_client = boto3.client(
        service_name="bedrock-runtime",
        region_name=AWS_REGION,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    print(f"Configured Bedrock client with model: {BEDROCK_MODEL_ID}")
else:
    bedrock_client = globals()['bedrock_client']
    BEDROCK_MODEL_ID = globals().get('BEDROCK_MODEL_ID', 'us.amazon.nova-2-lite-v1:0')
    print(f"Re-using existing Amazon Bedrock client ({BEDROCK_MODEL_ID}).")

# ==============================================================================
# 2. HYBRID GRAPH + VECTOR RETRIEVER (WITH CYPHER 25 + FALLBACK)
# ==============================================================================
YEAR_QUERY_RE = re.compile(r"\b(\d{3,4})\b")

def retrieve_graph_with_subgraph(question: str, top_k_pages: int = 4) -> dict:
    query_vec = embed_model.encode(question, normalize_embeddings=True).tolist()
    candidate_years = [int(y) for y in YEAR_QUERY_RE.findall(question)]

    with driver.session() as session:
        # Path A: Exact :ENTITIES Year Match
        entity_hits = []
        if candidate_years:
            entity_hits = session.run(
                """
                MATCH (e:ENTITIES)<-[:MENTIONS_ENTITY]-(p:Page)<-[:CONTAINS_PAGE]-(c:Chunk)
                WHERE e.year_value IN $years
                RETURN DISTINCT
                  p.node_id AS page_id,
                  p.page_number AS page_number,
                  c.node_id AS chunk_id,
                  1.0 AS score,
                  'Entity Match (' + e.name + ')' AS retrieval_source
                """,
                years=candidate_years,
            ).data()

        # Path B: Page Vector Search (Cypher 25 SEARCH with automatic fallback)
        try:
            vector_hits = session.run(
                """
                CYPHER 25
                MATCH (p:Page)
                SEARCH p IN (VECTOR INDEX page_embeddings FOR $query_vector LIMIT $top_k) SCORE AS score
                MATCH (c:Chunk)-[:CONTAINS_PAGE]->(p)
                RETURN
                  p.node_id AS page_id,
                  p.page_number AS page_number,
                  c.node_id AS chunk_id,
                  round(score, 4) AS score,
                  'Vector Search' AS retrieval_source
                """,
                top_k=top_k_pages,
                query_vector=query_vec,
            ).data()
        except Exception:
            # Fallback to standard vector query if DBMS doesn't support CYPHER 25
            vector_hits = session.run(
                """
                CALL db.index.vector.queryNodes('page_embeddings', $top_k, $query_vector)
                YIELD node AS p, score
                MATCH (c:Chunk)-[:CONTAINS_PAGE]->(p)
                RETURN
                  p.node_id AS page_id,
                  p.page_number AS page_number,
                  c.node_id AS chunk_id,
                  round(score, 4) AS score,
                  'Vector Search' AS retrieval_source
                """,
                top_k=top_k_pages,
                query_vector=query_vec,
            ).data()

        # Deduplicate seed pages
        seen_pages = {}
        for hit in entity_hits + vector_hits:
            pid = hit["page_id"]
            if pid not in seen_pages:
                seen_pages[pid] = hit
            else:
                seen_pages[pid]["retrieval_source"] += f" + {hit['retrieval_source']}"

        seed_page_ids = list(seen_pages.keys())[:5]

        # Path C: 2-Hop Expansion for LLM Context + Visual Subgraph
        expanded_records = session.run(
            """
            MATCH (c:Chunk)-[:CONTAINS_PAGE]->(p:Page)
            WHERE p.node_id IN $seed_ids
            OPTIONAL MATCH (p)-[r1:MENTIONS_ENTITY]->(e:ENTITIES)
            OPTIONAL MATCH (c)-[:CONTAINS_PAGE]->(neighbor:Page)
            WHERE abs(neighbor.page_number - p.page_number) <= 2 AND neighbor.has_text = true
            OPTIONAL MATCH (neighbor)-[r2:MENTIONS_ENTITY]->(ne:ENTITIES)
            WITH c, p,
                 collect(DISTINCT {
                   id: e.node_id, name: e.name, calendar: e.calendar,
                   snippet: r1.contexts[0]
                 }) AS page_entities,
                 collect(DISTINCT {
                   id: neighbor.node_id, page: neighbor.page_number, text: neighbor.text
                 }) AS nearby_pages,
                 collect(DISTINCT {
                   page_id: neighbor.node_id, id: ne.node_id, name: ne.name,
                   calendar: ne.calendar, snippet: r2.contexts[0]
                 }) AS neighbor_entities
            RETURN
              p.node_id AS page_id,
              p.page_number AS page_number,
              c.node_id AS chunk_id,
              toString(c.start_page) + '-' + toString(c.end_page) AS chunk_window,
              p.text AS page_text,
              page_entities,
              nearby_pages,
              neighbor_entities
            ORDER BY p.page_number ASC
            """,
            seed_ids=seed_page_ids,
        ).data()

    return {
        "seed_hits": list(seen_pages.values()),
        "expanded_pages": expanded_records,
    }

# ==============================================================================
# 3. INTERACTIVE SUBGRAPH RENDERER (PYVIS WITH HTML TOOLTIPS)
# ==============================================================================
def render_evidence_subgraph(question: str, retrieved: dict):
    net = Network(
        height="540px",
        width="100%",
        bgcolor="#0f172a",
        font_color="#f8fafc",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )

    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "centralGravity": 0.015,
          "springLength": 125,
          "springConstant": 0.07,
          "damping": 0.45
        },
        "solver": "forceAtlas2Based",
        "stabilization": { "iterations": 150 }
      },
      "interaction": { "hover": true, "tooltipDelay": 80, "navigationButtons": true }
    }
    """)

    seed_lookup = {h["page_id"]: h for h in retrieved["seed_hits"]}
    added_nodes = set()
    added_edges = set()

    # Central User Question Node
    short_q = question if len(question) <= 44 else question[:41] + "..."
    net.add_node(
        "USER_QUERY",
        label=f"User Question:\n{short_q}",
        title=f"<b>User Question:</b><br>{question}",
        color="#f59e0b",
        shape="star",
        size=30,
        font={"size": 13, "color": "#fef3c7", "bold": True},
    )
    added_nodes.add("USER_QUERY")

    for item in retrieved["expanded_pages"]:
        c_id = item["chunk_id"]
        p_id = item["page_id"]
        p_num = item["page_number"]

        # Chunk Node
        if c_id not in added_nodes:
            net.add_node(
                c_id,
                label=f"{c_id}\n(pp.{item['chunk_window']})",
                title=f"<b>Label:</b> :Chunk<br><b>ID:</b> {c_id}<br><b>Window:</b> Pages {item['chunk_window']}",
                color="#3b82f6",
                shape="box",
                size=22,
                font={"size": 12, "color": "#ffffff"},
            )
            added_nodes.add(c_id)

        # Primary Seed Page Node
        if p_id not in added_nodes:
            hit_info = seed_lookup.get(p_id, {})
            snippet = item["page_text"][:200].replace("\n", " ")
            net.add_node(
                p_id,
                label=f"Page {p_num}\n(Seed Hit)",
                title=(
                    f"<b>Label:</b> :Page (Primary Seed)<br>"
                    f"<b>Matched Via:</b> {hit_info.get('retrieval_source', 'Graph')}<br>"
                    f"<b>Score:</b> {hit_info.get('score', 'N/A')}<br>"
                    f"<b>Excerpt:</b> {snippet}..."
                ),
                color="#10b981",
                shape="dot",
                size=20,
                font={"size": 12, "color": "#ecfdf5", "bold": True},
            )
            added_nodes.add(p_id)

        if ("USER_QUERY", p_id) not in added_edges:
            net.add_edge("USER_QUERY", p_id, label="MATCHED", color="#fbbf24", dashes=True, width=2.0)
            added_edges.add(("USER_QUERY", p_id))

        if (c_id, p_id) not in added_edges:
            net.add_edge(c_id, p_id, label="CONTAINS_PAGE", color="#60a5fa", width=1.5)
            added_edges.add((c_id, p_id))

        # 2-Hop Neighbor Pages in same Chunk
        for nb in item["nearby_pages"]:
            nb_id = nb["id"]
            if not nb_id or not nb["text"].strip():
                continue
            if nb_id not in added_nodes:
                snippet = nb["text"][:200].replace("\n", " ")
                net.add_node(
                    nb_id,
                    label=f"Page {nb['page']}\n(2-Hop)",
                    title=(
                        f"<b>Label:</b> :Page (2-Hop Context Neighbor)<br>"
                        f"<b>Parent Chunk:</b> {c_id}<br>"
                        f"<b>Excerpt:</b> {snippet}..."
                    ),
                    color="#14b8a6",
                    shape="dot",
                    size=14,
                    font={"size": 10, "color": "#cbd5e1"},
                )
                added_nodes.add(nb_id)
            if (c_id, nb_id) not in added_edges:
                net.add_edge(c_id, nb_id, label="CONTAINS_PAGE", color="#2dd4bf", width=1.1)
                added_edges.add((c_id, nb_id))

        # Connected :ENTITIES (A.D. and A.H. Years)
        all_ents = [(p_id, e) for e in item["page_entities"] if e["id"]] + [
            (ne["page_id"], ne) for ne in item["neighbor_entities"] if ne["id"] and ne["page_id" ] in added_nodes
        ]
        for src_page_id, ent in all_ents:
            e_id = ent["id"]
            if e_id not in added_nodes:
                is_ah = ent["calendar"] == "AH"
                net.add_node(
                    e_id,
                    label=ent["name"],
                    title=(
                        f"<b>Label:</b> :ENTITIES (Year)<br>"
                        f"<b>Value:</b> {ent['name']}<br>"
                        f"<b>Calendar:</b> {ent['calendar']}"
                    ),
                    color="#c084fc" if is_ah else "#fb7185",
                    shape="ellipse",
                    size=18,
                    font={"size": 12, "color": "#ffffff", "bold": True},
                )
                added_nodes.add(e_id)
            if (src_page_id, e_id) not in added_edges:
                ctx = ent.get("snippet") or ""
                net.add_edge(
                    src_page_id,
                    e_id,
                    label="MENTIONS_ENTITY",
                    title=f"<b>[:MENTIONS_ENTITY]</b><br>{ctx}",
                    color="#fda4af",
                    width=1.8,
                )
                added_edges.add((src_page_id, e_id))

    html_str = net.generate_html()
    tooltip_fix = """
    <style>
      div.vis-tooltip {
        background-color: #1e293b !important; color: #f8fafc !important;
        border: 1px solid #475569 !important; border-radius: 8px !important;
        padding: 10px 14px !important; font-family: sans-serif !important;
        font-size: 13px !important; line-height: 1.45 !important;
        max-width: 360px !important; white-space: normal !important;
      }
    </style>
    <script>
      (function() {
        function toEl(h) { const d = document.createElement("div"); d.innerHTML = h; return d; }
        if (typeof nodes !== "undefined") nodes.forEach(n => { if (typeof n.title === "string") nodes.update({id: n.id, title: toEl(n.title)}); });
        if (typeof edges !== "undefined") edges.forEach(e => { if (typeof e.title === "string") edges.update({id: e.id, title: toEl(e.title)}); });
      })();
    </script></body>
    """
    html_str = html_str.replace("</body>", tooltip_fix)
    with open("latest_query_subgraph.html", "w", encoding="utf-8") as f:
        f.write(html_str)
    display(HTML(html_str))

# ==============================================================================
# 4. END-TO-END Q&A + GRAPH VISUALIZER FUNCTION
# ==============================================================================
def run_qna_with_graph(question: str):
    t0 = time.time()
    retrieved = retrieve_graph_with_subgraph(question)

    context_blocks = []
    included_page_nums = set()

    for item in retrieved["expanded_pages"]:
        p_num = item["page_number"]
        included_page_nums.add(p_num)
        years_str = ", ".join(
            sorted({f"{e['name']} [{e['calendar']}]" for e in item["page_entities"] if e["id"]})
        ) or "None"
        context_blocks.append(
            f"=== [PRIMARY MATCH: Page {p_num} | {item['chunk_id']} (pp.{item['chunk_window']}) | "
            f"Graph Year Entities: {years_str}] ===\n{item['page_text']}"
        )

    for item in retrieved["expanded_pages"]:
        for nb in item["nearby_pages"]:
            nb_num = nb["page"]
            if nb_num not in included_page_nums and nb["text"].strip():
                included_page_nums.add(nb_num)
                context_blocks.append(
                    f"=== [2-HOP NEIGHBOR PAGE: Page {nb_num} | {item['chunk_id']}] ===\n{nb['text']}"
                )

    formatted_context = "\n\n".join(context_blocks)

    system_prompts = [{
        "text": (
            "You are an expert architectural historian analyzing 'The Antiquities of Bidar' (1917) by G. Yazdani. "
            "Answer strictly using ONLY the provided Neo4j GraphRAG context. "
            "Cite exact page numbers, chunk IDs, and year entities (both A.D. and A.H. when present) "
            "in brackets like [Page 28 | chunk_06 | 1472 A.D. / 877 A.H.]."
        )
    }]

    response = bedrock_client.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": f"CONTEXT:\n{formatted_context}\n\nQUESTION:\n{question}"}]}],
        system=system_prompts,
        inferenceConfig={"temperature": 0.2, "maxTokens": 1024},
    )

    print("=" * 80)
    print(f"QUESTION: {question}")
    print(f"Latency: {time.time() - t0:.2f}s")
    print("=" * 80)
    display(Markdown(response["output"]["message"]["content"][0]["text"]))
    print("\nSUBGRAPH EVIDENCE MAP (★ Query | ■ Chunk | ● Green=Seed Page | ● Teal=2-Hop Page | ⬭ Coral=A.D. | ⬭ Purple=A.H.):")
    render_evidence_subgraph(question, retrieved)

# Test Q&A with Interactive Subgraph Mapping
run_qna_with_graph("Tell me about the Tomb of Ali Barid, including its dates in A.H. and A.D. and architectural features.")

Re-using existing Neo4j connection.
Re-using loaded sentence embedding model.
Re-using existing Amazon Bedrock client (us.amazon.nova-2-lite-v1:0).
QUESTION: Tell me about the Tomb of Ali Barid, including its dates in A.H. and A.D. and architectural features.
Latency: 3.47s


The Tomb of 'Ali Barid, the third ruler of the Barid Shahi line, is an imposing structure situated on an elevated plain to the west of Bidar town [Page 31 | chunk_07]. Originally surrounded by elegant gardens with water channels and fountains, the tomb now stands amidst ruins [Page 31 | chunk_07]. 

**Architectural Features:**
- **Plan and Structure:** The tomb is square in plan, measuring 57 feet on each side, with an open arch on all four sides [Page 31 | chunk_07]. 
- **Dome:** It is crowned by a turnip-shaped dome terminating in a spire, with a total height of 106 feet from the ground [Page 31 | chunk_07]. 
- **Interior:** The interior is highly ornamented, featuring several courses of exquisite enamel-work arranged in calligraphical devices along the walls [Page 31 | chunk_07]. Unlike other Muhammadan tombs, it allows plenty of light and air through its large side openings [Page 31 | chunk_07].
- **Construction Materials:** The tomb is built of hard black trap up to the drum of the dome, with a brick and lime top, likely to protect the building from extra weight [Page 31 | chunk_07].

**Dates:**
- The tomb's dome is dated to **984 A.H. (1576 A.D.)** [Page 33 | chunk_07].
- The death of 'Ali Barid is dated to **987 A.H. (1579 A.D.)** [Page 33 | chunk_07].

The tomb's perfect proportions and elegant design make it a notable example of Barid Shahi architecture [Page 31 | chunk_07].


SUBGRAPH EVIDENCE MAP (★ Query | ■ Chunk | ● Green=Seed Page | ● Teal=2-Hop Page | ⬭ Coral=A.D. | ⬭ Purple=A.H.):


### Dynamic Runtime Q&A Interface
Type any question regarding the Antiquities of Bidar in the box below and click the button to trigger a dynamic hybrid retrieval, Bedrock LLM generation, and interactive PyVis network visualization.

In [16]:
import ipywidgets as widgets
from IPython.display import clear_output, display, Markdown

# Define UI Widgets
question_input = widgets.Textarea(
    value="Who built the Madrasah of Mahmud Gawan and when was it struck by lightning?",
    placeholder='Type your historical query here...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='80px')
)

submit_button = widgets.Button(
    description='Run GraphRAG Q&A',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='200px', height='40px')
)

output_container = widgets.Output()

def on_submit_clicked(b):
    with output_container:
        clear_output(wait=True)
        user_query = question_input.value.strip()
        if not user_query:
            print("Please enter a valid question first.")
            return

        print(f"Searching Graph Index and Synthesizing Answer...")
        try:
            run_qna_with_graph(user_query)
        except Exception as e:
            print(f"An error occurred during execution: {e}")

submit_button.on_click(on_submit_clicked)

# Layout Arrangement
ui_layout = widgets.VBox([
    question_input,
    widgets.Box([submit_button], layout=widgets.Layout(justify_content='flex-end', margin='10px 0px')),
    output_container
])

display(ui_layout)

In [17]:
import re
import numpy as np
import pandas as pd
from getpass import getpass
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# 1. Reuse existing Neo4j driver & embedding model (or reconnect if kernel restarted)
if "driver" not in globals():
    NEO4J_URI = "neo4j+s://c7554cbf.databases.neo4j.io"
    NEO4J_USERNAME = "neo4j"
    try:
        from google.colab import userdata
        NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")
    except Exception:
        NEO4J_PASSWORD = None
    if not NEO4J_PASSWORD:
        NEO4J_PASSWORD = getpass("Enter NEO4J_PASSWORD: ").strip()
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()

if "embed_model" not in globals():
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Define the 7 Structured Themes
THEMES_CONFIG = [
    {
        "code": "AGRICULTURE_CROPS",
        "name": "Agriculture & Land Use",
        "status": "Active",
        "description": "Crops, yields, cultivation, land tenure, and forest land.",
        "keywords": ["jowar", "cotton", "rice", "crop", "cultivation", "tenure", "forest"],
        "color": "#22c55e",  # Emerald Green
    },
    {
        "code": "CLIMATE_METRICS",
        "name": "Climate & Heat",
        "status": "Active",
        "description": "Temperature, rainfall, monsoon trends, drought, humidity, and heat waves.",
        "keywords": ["temperature", "rainfall", "monsoon", "drought", "humidity", "heat wave"],
        "color": "#f97316",  # Orange
    },
    {
        "code": "INEQUALITY_LABOUR",
        "name": "Socio-Economic Inequality",
        "status": "Active",
        "description": "Caste, gendered labour, wages, migration, housing, and ventilation.",
        "keywords": ["caste", "labour", "wage", "migration", "housing", "ventilation"],
        "color": "#eab308",  # Amber Yellow
    },
    {
        "code": "IRRIGATION_WATER",
        "name": "Irrigation & Water",
        "status": "Active",
        "description": "Tanks, wells, canals, reservoirs, groundwater, and irrigation projects.",
        "keywords": ["tank", "well", "canal", "reservoir", "groundwater", "irrigation"],
        "color": "#06b6d4",  # Cyan
    },
    {
        "code": "LOCATION",
        "name": "Location & Administrative Units",
        "status": "Active",
        "description": "Districts, taluks, villages, towns, and boundary changes.",
        "keywords": ["district", "taluk", "village", "town", "boundary"],
        "color": "#3b82f6",  # Royal Blue
    },
    {
        "code": "POLICIES_SCHEMES",
        "name": "Government & Policies",
        "status": "Active",
        "description": "Acts, relief schemes, committees, and administrative programmes.",
        "keywords": ["act", "scheme", "committee", "relief", "government", "policy"],
        "color": "#a855f7",  # Violet
    },
    {
        "code": "PUBLIC_HEALTH",
        "name": "Health & Sanitation",
        "status": "Active",
        "description": "Disease, sanitation, water contamination, and public-health impacts.",
        "keywords": ["cholera", "malaria", "plague", "sanitation", "contamination", "sewage"],
        "color": "#ef4444",  # Crimson Red
    },
]

# Pre-compile word-boundary regexes (handles plurals like boundary->boundaries, tank->tanks, act->acts)
def build_kw_regex(kw: str) -> re.Pattern:
    escaped = re.escape(kw.lower())
    if escaped.endswith("y"):
        stem = escaped[:-1]
        pattern = rf"\b{stem}(?:y|ies)\b"
    else:
        pattern = rf"\b{escaped}(?:s|es)?\b"
    return re.compile(pattern, re.IGNORECASE)

THEME_REGEXES = {
    t["code"]: [(kw, build_kw_regex(kw)) for kw in t["keywords"]]
    for t in THEMES_CONFIG
}

# Pre-compute theme embeddings for hybrid semantic matching
theme_texts = [
    f"{t['name']}: {t['description']} Keywords: {', '.join(t['keywords'])}"
    for t in THEMES_CONFIG
]
theme_vecs = embed_model.encode(theme_texts, normalize_embeddings=True)
THEME_EMBEDDINGS = {
    t["code"]: vec for t, vec in zip(THEMES_CONFIG, theme_vecs)
}

# 3. Fetch Active Pages from Neo4j Aura & Score Against All 7 Themes
with driver.session() as session:
    session.run(
        "CREATE CONSTRAINT theme_code_unique IF NOT EXISTS FOR (t:THEME) REQUIRE t.code IS UNIQUE"
    )

    # Upsert :THEME nodes
    session.run(
        """
        UNWIND $themes AS t
        MERGE (th:THEME {code: t.code})
        SET th.node_id = 'theme_' + t.code,
            th.name = t.name,
            th.status = t.status,
            th.description = t.description,
            th.keywords = t.keywords,
            th.color = t.color
        """,
        themes=THEMES_CONFIG,
    )

    pages = session.run(
        """
        MATCH (p:Page)
        WHERE p.has_text = true AND size(trim(p.text)) > 0
        RETURN p.node_id AS page_id, p.page_number AS page_number, p.text AS text, p.embedding AS embedding
        ORDER BY p.page_number ASC
        """
    ).data()

page_theme_edges = []
SEMANTIC_THRESHOLD = 0.36  # Captures strong topical alignment even when exact keyword differs

for p in pages:
    text = p["text"]
    p_emb = np.array(p["embedding"], dtype=np.float32) if p.get("embedding") else None

    for t in THEMES_CONFIG:
        code = t["code"]
        matched_kws = [kw for kw, rx in THEME_REGEXES[code] if rx.search(text)]
        sem_score = float(np.dot(p_emb, THEME_EMBEDDINGS[code])) if p_emb is not None else 0.0

        if matched_kws or sem_score >= SEMANTIC_THRESHOLD:
            match_type = (
                "Keyword + Semantic"
                if (matched_kws and sem_score >= SEMANTIC_THRESHOLD)
                else ("Keyword" if matched_kws else "Semantic")
            )
            page_theme_edges.append({
                "page_id": p["page_id"],
                "theme_code": code,
                "matched_keywords": matched_kws,
                "semantic_score": round(sem_score, 4),
                "match_type": match_type,
            })

# 4. Write (:Page)-[:HAS_THEME]->(:THEME) and (:ENTITIES)-[:RELATED_TO_THEME]->(:THEME)
with driver.session() as session:
    # Clear old theme edges if re-running
    session.run("MATCH ()-[r:HAS_THEME|RELATED_TO_THEME]->() DELETE r")

    session.run(
        """
        UNWIND $rows AS row
        MATCH (p:Page {node_id: row.page_id})
        MATCH (th:THEME {code: row.theme_code})
        MERGE (p)-[r:HAS_THEME]->(th)
        SET r.matched_keywords = row.matched_keywords,
            r.semantic_score = row.semantic_score,
            r.match_type = row.match_type
        """,
        rows=page_theme_edges,
    )

    # Connect Year :ENTITIES to :THEME whenever they co-occur on the same page
    session.run(
        """
        MATCH (e:ENTITIES)<-[:MENTIONS_ENTITY]-(p:Page)-[r:HAS_THEME]->(th:THEME)
        WITH e, th,
             collect(DISTINCT p.page_number) AS shared_pages,
             reduce(acc = [], kws IN collect(r.matched_keywords) | acc + kws) AS all_kws,
             max(r.semantic_score) AS max_sem_score
        MERGE (e)-[rel:RELATED_TO_THEME]->(th)
        SET rel.pages = shared_pages,
            rel.page_count = size(shared_pages),
            rel.keywords = apoc.coll.toSet(all_kws),
            rel.max_semantic_score = max_sem_score
        """
    )

    # Audit summary table
    audit_rows = session.run(
        """
        MATCH (th:THEME)
        OPTIONAL MATCH (p:Page)-[:HAS_THEME]->(th)
        OPTIONAL MATCH (e:ENTITIES)-[:RELATED_TO_THEME]->(th)
        RETURN
          th.code AS ThemeCode,
          th.name AS ThemeName,
          count(DISTINCT p) AS LinkedPages,
          count(DISTINCT e) AS LinkedYears,
          collect(DISTINCT e.name)[0..6] AS SampleYears
        ORDER BY LinkedPages DESC
        """
    ).data()

print("=== NEO4J AURA THEME INDEXING COMPLETE ===")
display(pd.DataFrame(audit_rows))

=== NEO4J AURA THEME INDEXING COMPLETE ===


,ThemeCode,ThemeName,LinkedPages,LinkedYears,SampleYears
0,LOCATION,Location & Administrative Units,11,16,"[1347 A.D., 1429 A.D., 1432 A.D., 1435 A.D., 1..."
1,POLICIES_SCHEMES,Government & Policies,6,11,"[1432 A.D., 1435 A.D., 1459 A.D., 1463 A.D., 1..."
2,IRRIGATION_WATER,Irrigation & Water,4,5,"[1347 A.D., 1426 A.D., 1429 A.D., 1667 A.D., 8..."
3,AGRICULTURE_CROPS,Agriculture & Land Use,0,0,[]
4,CLIMATE_METRICS,Climate & Heat,0,0,[]
5,INEQUALITY_LABOUR,Socio-Economic Inequality,0,0,[]
6,PUBLIC_HEALTH,Health & Sanitation,0,0,[]


In [18]:
import html as html_lib
import json
import re
import time
import boto3
import ipywidgets as widgets
from getpass import getpass
from neo4j import GraphDatabase
from pyvis.network import Network
from IPython.display import clear_output, display, HTML, Markdown

# ==============================================================================
# 1. VERIFY ACTIVE CONNECTIONS (NEO4J AURA + AMAZON BEDROCK)
# ==============================================================================
if "driver" not in globals():
    NEO4J_URI = "neo4j+s://c7554cbf.databases.neo4j.io"
    NEO4J_USERNAME = "neo4j"
    NEO4J_PASSWORD = getpass("Enter NEO4J_PASSWORD: ").strip()
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()

if "bedrock_client" not in globals():
    AWS_REGION = input("Enter AWS Region [default: us-east-1]: ").strip() or "us-east-1"
    AWS_ACCESS_KEY_ID = getpass("Enter AWS_ACCESS_KEY_ID: ").strip()
    AWS_SECRET_ACCESS_KEY = getpass("Enter AWS_SECRET_ACCESS_KEY: ").strip()
    BEDROCK_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
    bedrock_client = boto3.client(
        service_name="bedrock-runtime",
        region_name=AWS_REGION,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )

# ==============================================================================
# 2. ISOLATED IFRAME RENDERER (PREVENTS MULTIPLE PYVIS GRAPHS COLLIDING)
# ==============================================================================
def render_pyvis_in_colab(net: Network, height_px: int = 530):
    html_str = net.generate_html()
    tooltip_fix = """
    <style>
      html, body { margin: 0; padding: 0; overflow: hidden; background-color: #0f172a; }
      div.vis-tooltip {
        background-color: #1e293b !important; color: #f8fafc !important;
        border: 1px solid #475569 !important; border-radius: 8px !important;
        padding: 10px 14px !important; font-family: -apple-system, BlinkMacSystemFont, sans-serif !important;
        font-size: 13px !important; line-height: 1.45 !important;
        max-width: 380px !important; white-space: normal !important;
        box-shadow: 0 10px 25px rgba(0, 0, 0, 0.55) !important;
      }
    </style>
    <script>
      (function() {
        function toEl(h) { const d = document.createElement("div"); d.innerHTML = h; return d; }
        if (typeof nodes !== "undefined") nodes.forEach(n => { if (typeof n.title === "string") nodes.update({id: n.id, title: toEl(n.title)}); });
        if (typeof edges !== "undefined") edges.forEach(e => { if (typeof e.title === "string") edges.update({id: e.id, title: toEl(e.title)}); });
      })();
    </script></body>
    """
    html_str = html_str.replace("</body>", tooltip_fix)
    escaped_html = html_lib.escape(html_str, quote=True)
    iframe_tag = (
        f'<iframe srcdoc="{escaped_html}" '
        f'style="width:100%; height:{height_px}px; border:1px solid #334155; border-radius:8px; background:#0f172a;">'
        f'</iframe>'
    )
    display(HTML(iframe_tag))

# ==============================================================================
# 3. STEP 1 GRAPH: DOC-ROOT -> ALL 42 EXTRACTED YEARS HIERARCHY
# ==============================================================================
def render_doc_root_years_tree():
    with driver.session() as session:
        doc = session.run("MATCH (d:Document) RETURN d.node_id AS id, d.title AS title").single()
        years = session.run(
            """
            MATCH (e:ENTITIES)
            OPTIONAL MATCH (e)-[r:RELATED_TO_THEME]->(th:THEME)
            RETURN
              e.node_id AS id,
              e.name AS name,
              e.calendar AS calendar,
              e.year_value AS year_val,
              e.pages AS pages,
              e.chunks AS chunks,
              collect(DISTINCT th.name) AS linked_themes
            ORDER BY e.calendar ASC, e.year_value ASC
            """
        ).data()

    net = Network(
        height="470px",
        width="100%",
        bgcolor="#0f172a",
        font_color="#f8fafc",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )

    net.set_options("""
    {
      "layout": {
        "hierarchical": {
          "enabled": true,
          "direction": "UD",
          "sortMethod": "directed",
          "nodeSpacing": 95,
          "levelSeparation": 170
        }
      },
      "physics": { "enabled": false },
      "interaction": { "hover": true, "tooltipDelay": 80, "navigationButtons": true }
    }
    """)

    doc_id = doc["id"]
    net.add_node(
        doc_id,
        label=f"DOC-ROOT\n{doc['title']}",
        title=f"<b>Root Document:</b> {doc['title']}<br><b>Child Year Entities:</b> {len(years)}",
        color="#f59e0b",
        shape="star",
        size=34,
        level=0,
        font={"size": 14, "color": "#fef3c7", "bold": True},
    )

    for y in years:
        is_ah = y["calendar"] == "AH"
        themes = [t for t in y["linked_themes"] if t]
        themes_html = "<br>• ".join(themes) if themes else "<i>No direct keyword theme hit</i>"

        net.add_node(
            y["id"],
            label=y["name"],
            title=(
                f"<b>Year Entity:</b> {y['name']} ({y['calendar']})<br>"
                f"<b>Pages:</b> {y['pages']} | <b>Chunks:</b> {', '.join(y['chunks'])}<br>"
                f"<b>Linked Themes ({len(themes)}):</b><br>• {themes_html}"
            ),
            color="#c084fc" if is_ah else "#fb7185",
            shape="ellipse",
            size=16 + (len(themes) * 2),
            level=1,
            borderWidth=3 if themes else 1,
            font={"size": 11, "color": "#ffffff", "bold": bool(themes)},
        )
        net.add_edge(
            doc_id,
            y["id"],
            label="HAS_YEAR",
            color="#475569",
            width=1.2,
            font={"size": 8, "color": "#64748b", "strokeWidth": 0},
        )

    render_pyvis_in_colab(net, height_px=490)
    return years

# ==============================================================================
# 4. STEP 2 GRAPH & FACTS: SELECTED YEAR -> PAGES -> THEMES & FACTS
# ==============================================================================
def explore_year_themes(year_entity_id: str, selected_theme_code: str = "ALL"):
    t0 = time.time()

    with driver.session() as session:
        records = session.run(
            """
            MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)-[:CONTAINS_PAGE]->(p:Page)-[m:MENTIONS_ENTITY]->(e:ENTITIES {node_id: $eid})
            OPTIONAL MATCH (p)-[ht:HAS_THEME]->(th:THEME)
            OPTIONAL MATCH (c)-[:CONTAINS_PAGE]->(nb:Page)
            WHERE abs(nb.page_number - p.page_number) <= 1 AND nb.has_text = true
            RETURN
              d.node_id AS doc_id,
              d.title AS doc_title,
              e.node_id AS year_id,
              e.name AS year_name,
              e.calendar AS calendar,
              c.node_id AS chunk_id,
              toString(c.start_page) + '-' + toString(c.end_page) AS chunk_window,
              p.node_id AS page_id,
              p.page_number AS page_number,
              p.text AS page_text,
              m.contexts AS mention_snippets,
              collect(DISTINCT {
                code: th.code,
                name: th.name,
                color: th.color,
                keywords: ht.matched_keywords,
                sem_score: ht.semantic_score,
                match_type: ht.match_type
              }) AS graph_themes,
              collect(DISTINCT {
                page_id: nb.node_id,
                page_number: nb.page_number,
                text: nb.text
              }) AS neighbor_pages
            ORDER BY p.page_number ASC
            """,
            eid=year_entity_id,
        ).data()

        all_themes_meta = session.run(
            "MATCH (th:THEME) RETURN th.code AS code, th.name AS name, th.description AS desc, th.keywords AS keywords, th.color AS color"
        ).data()

    if not records:
        print("No records found for the selected year.")
        return

    year_name = records[0]["year_name"]
    doc_id = records[0]["doc_id"]
    doc_title = records[0]["doc_title"]

    context_blocks = []
    seen_pages = set()
    for r in records:
        p_num = r["page_number"]
        seen_pages.add(p_num)
        context_blocks.append(
            f"=== [PRIMARY YEAR PAGE: Page {p_num} | {r['chunk_id']} (pp.{r['chunk_window']}) | Year: {year_name}] ===\n{r['page_text']}"
        )
    for r in records:
        for nb in r["neighbor_pages"]:
            if nb["page_number"] not in seen_pages and nb["text"].strip():
                seen_pages.add(nb["page_number"])
                context_blocks.append(
                    f"=== [ADJACENT CONTEXT PAGE: Page {nb['page_number']} | {r['chunk_id']}] ===\n{nb['text']}"
                )

    formatted_context = "\n\n".join(context_blocks)

    target_themes = (
        all_themes_meta
        if selected_theme_code == "ALL"
        else [t for t in all_themes_meta if t["code"] == selected_theme_code]
    )
    theme_spec_str = "\n".join(
        f"- {t['code']} ({t['name']}): {t['desc']} [Keywords: {', '.join(t['keywords'])}]"
        for t in target_themes
    )

    system_prompts = [{
        "text": (
            "You are a historical data extraction engine analyzing 'The Antiquities of Bidar' (1917). "
            "Given text pages linked to a specific historical year, extract ONLY factual evidence relevant to each listed Theme.\n"
            "Rules:\n"
            "1. Return STRICT valid JSON matching this schema:\n"
            '{"themes": {"THEME_CODE": {"has_evidence": true/false, "facts": ["fact 1 [Page X | chunk_Y]", ...], "matched_concepts": ["kw1", ...]}}}\n'
            "2. NEVER invent or extrapolate facts not explicitly stated in the provided context.\n"
            "3. If a theme has no evidence in the provided pages, set \"has_evidence\": false and \"facts\": []."
        )
    }]

    user_prompt = (
        f"SELECTED YEAR: {year_name}\n\n"
        f"THEMES TO EXTRACT:\n{theme_spec_str}\n\n"
        f"RETRIEVED PAGE CONTEXT:\n{formatted_context}"
    )

    response = bedrock_client.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": user_prompt}]}],
        system=system_prompts,
        inferenceConfig={"temperature": 0.1, "maxTokens": 1200},
    )

    raw_llm_text = response["output"]["message"]["content"][0]["text"]
    cleaned_json = re.sub(r"^```(?:json)?|```$", "", raw_llm_text.strip(), flags=re.MULTILINE).strip()

    try:
        extracted_data = json.loads(cleaned_json).get("themes", {})
    except Exception:
        extracted_data = {}

    elapsed = round(time.time() - t0, 2)

    # Display Thematic Facts Table
    display(Markdown(f"## Year Selected: `{year_name}` *(Analyzed in {elapsed}s)*"))

    md_lines = ["| Theme | Code | Status | Extracted Historical Facts & Citations |", "| :--- | :--- | :--- | :--- |"]
    active_theme_facts = {}

    for t in target_themes:
        code = t["code"]
        info = extracted_data.get(code, {})
        facts = info.get("facts", [])
        has_ev = bool(info.get("has_evidence") and facts)

        if has_ev:
            active_theme_facts[code] = facts
            facts_bullets = "<br>".join(f"• {f}" for f in facts)
            md_lines.append(f"| **{t['name']}** | `{code}` | Active Evidence | {facts_bullets} |")
        else:
            md_lines.append(f"| {t['name']} | `{code}` | *No Evidence* | *No direct facts for this theme on {year_name} pages.* |")

    display(Markdown("\n".join(md_lines)))

    # Render Interactive Visual Graph inside its own isolated iframe
    display(Markdown(
        "---\n### Interactive Visual Graph (`Doc-Root` → `Selected Year` → `Pages` → `Themes & Facts`)\n"
        "`★ Gold = Doc-Root` | `⬭ Coral/Purple = Selected Year` | `■ Blue = Chunk` | "
        "`● Green = Year Page` | `◆ Colored Diamonds = Matched Themes (Hover to read extracted facts!)`"
    ))

    net = Network(
        height="520px",
        width="100%",
        bgcolor="#0f172a",
        font_color="#f8fafc",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )

    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -85,
          "centralGravity": 0.015,
          "springLength": 135,
          "springConstant": 0.06,
          "damping": 0.45
        },
        "solver": "forceAtlas2Based",
        "stabilization": { "iterations": 160 }
      },
      "interaction": { "hover": true, "tooltipDelay": 80, "navigationButtons": true }
    }
    """)

    added_nodes = set()
    added_edges = set()

    net.add_node(
        doc_id,
        label=f"DOC-ROOT\n{doc_title}",
        title=f"<b>Root Document:</b> {doc_title}",
        color="#f59e0b",
        shape="star",
        size=30,
        font={"size": 13, "color": "#fef3c7", "bold": True},
    )
    added_nodes.add(doc_id)

    y_id = records[0]["year_id"]
    is_ah = records[0]["calendar"] == "AH"
    net.add_node(
        y_id,
        label=f"Selected Year:\n{year_name}",
        title=f"<b>Selected Year:</b> {year_name}<br><b>Calendar:</b> {records[0]['calendar']}",
        color="#c084fc" if is_ah else "#fb7185",
        shape="ellipse",
        size=26,
        font={"size": 14, "color": "#ffffff", "bold": True},
    )
    added_nodes.add(y_id)
    net.add_edge(doc_id, y_id, label="HAS_YEAR", color="#fbbf24", width=2.2)

    theme_meta_lookup = {t["code"]: t for t in target_themes}

    for r in records:
        c_id = r["chunk_id"]
        p_id = r["page_id"]
        p_num = r["page_number"]

        if c_id not in added_nodes:
            net.add_node(
                c_id,
                label=f"{c_id}\n(pp.{r['chunk_window']})",
                title=f"<b>Chunk:</b> {c_id} (Pages {r['chunk_window']})",
                color="#3b82f6",
                shape="box",
                size=20,
                font={"size": 11, "color": "#ffffff"},
            )
            added_nodes.add(c_id)

        if p_id not in added_nodes:
            mentions_preview = "<br>• ".join(r["mention_snippets"] or [])
            net.add_node(
                p_id,
                label=f"Page {p_num}",
                title=(
                    f"<b>Page {p_num}</b> ({c_id})<br>"
                    f"<b>Year Mention Context:</b><br>• {mentions_preview}"
                ),
                color="#10b981",
                shape="dot",
                size=19,
                font={"size": 12, "color": "#ecfdf5", "bold": True},
            )
            added_nodes.add(p_id)

        if (y_id, p_id) not in added_edges:
            net.add_edge(y_id, p_id, label="APPEARS_IN_PAGE", color="#fda4af", width=2.0)
            added_edges.add((y_id, p_id))

        if (c_id, p_id) not in added_edges:
            net.add_edge(c_id, p_id, label="CONTAINS_PAGE", color="#60a5fa", width=1.4)
            added_edges.add((c_id, p_id))

        page_theme_codes = {gt["code"]: gt for gt in r["graph_themes"] if gt["code"] in theme_meta_lookup}
        for code in set(page_theme_codes.keys()) | set(active_theme_facts.keys()):
            t_meta = theme_meta_lookup.get(code)
            if not t_meta:
                continue
            th_node_id = f"theme_{code}"
            facts_list = active_theme_facts.get(code, [])
            gt_info = page_theme_codes.get(code, {})
            kws = ", ".join(gt_info.get("keywords") or []) or "Semantic / LLM Extraction"

            if th_node_id not in added_nodes:
                facts_html = (
                    "<br>• ".join(facts_list)
                    if facts_list
                    else "<i>Topical/keyword match on page; see table for details.</i>"
                )
                net.add_node(
                    th_node_id,
                    label=f"{t_meta['name']}\n[{code}]",
                    title=(
                        f"<b>Theme:</b> {t_meta['name']} ({code})<br>"
                        f"<b>Matched Keywords:</b> {kws}<br>"
                        f"<b>Extracted Facts:</b><br>• {facts_html}"
                    ),
                    color=t_meta["color"],
                    shape="diamond",
                    size=24 if facts_list else 17,
                    font={"size": 12, "color": "#ffffff", "bold": True},
                )
                added_nodes.add(th_node_id)

            if (p_id, th_node_id) not in added_edges:
                net.add_edge(
                    p_id,
                    th_node_id,
                    label="HAS_THEME",
                    title=f"<b>Keywords:</b> {kws}",
                    color=t_meta["color"],
                    width=2.0,
                )
                added_edges.add((p_id, th_node_id))

    render_pyvis_in_colab(net, height_px=540)

# ==============================================================================
# 5. INTERACTIVE WIDGET UI (DOC-ROOT -> YEAR SELECTOR -> THEMATIC FACTS + GRAPH)
# ==============================================================================
display(Markdown("## Step 1: `Doc-Root` → All Extracted Years Hierarchy"))
all_years_list = render_doc_root_years_tree()

year_options = []
default_year_id = all_years_list[0]["id"]
for y in all_years_list:
    t_count = len([t for t in y["linked_themes"] if t])
    label = f"{y['name']}  —  (Pages: {y['pages']} | Linked Themes: {t_count})"
    year_options.append((label, y["id"]))
    if y["id"] == "year_AD_1686":
        default_year_id = y["id"]

theme_filter_options = [("All 7 Themes (Complete Thematic Scan)", "ALL")] + [
    (f"{t['name']} ({t['code']})", t["code"]) for t in THEMES_CONFIG
]

year_dropdown = widgets.Dropdown(
    options=year_options,
    value=default_year_id,
    description="Select Year:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="58%"),
)

theme_dropdown = widgets.Dropdown(
    options=theme_filter_options,
    value="ALL",
    description="Theme Focus:",
    style={"description_width": "95px"},
    layout=widgets.Layout(width="40%"),
)

explore_button = widgets.Button(
    description="Show Thematic Facts & Visual Graph",
    button_style="primary",
    icon="sitemap",
    layout=widgets.Layout(width="280px", height="40px"),
)

theme_output_container = widgets.Output()

def on_explore_triggered(_=None):
    with theme_output_container:
        clear_output(wait=True)
        print("Traversing Neo4j Graph & Extracting Thematic Facts via Amazon Bedrock...")
        try:
            explore_year_themes(year_dropdown.value, theme_dropdown.value)
        except Exception as e:
            print(f"Error during thematic exploration: {e}")

explore_button.on_click(on_explore_triggered)
year_dropdown.observe(lambda change: on_explore_triggered() if change["name"] == "value" else None, names="value")

ui_layout = widgets.VBox([
    widgets.HTML("<h3>Step 2: Select a Child Year to Inspect Thematic Facts & Evidence Graph</h3>"),
    widgets.HBox([year_dropdown, theme_dropdown]),
    widgets.Box([explore_button], layout=widgets.Layout(justify_content="flex-end", margin="10px 0px")),
    theme_output_container,
])

display(ui_layout)
on_explore_triggered()

## Step 1: `Doc-Root` → All Extracted Years Hierarchy

/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [19]:
!pip install -q geopy neo4j boto3 pandas

import json
import re
import time
import boto3
import pandas as pd
from collections import defaultdict
from getpass import getpass
from neo4j import GraphDatabase
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ==============================================================================
# 1. REUSE OR INITIALIZE CONNECTIONS (NEO4J AURA + AMAZON BEDROCK)
# ==============================================================================
def _get_secret_or_prompt(key_name: str, prompt_text: str, is_secret: bool = True) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(key_name)
        if val:
            return val.strip()
    except Exception:
        pass
    return (getpass(prompt_text) if is_secret else input(prompt_text)).strip()

if "driver" not in globals():
    NEO4J_URI = "neo4j+s://c7554cbf.databases.neo4j.io"
    NEO4J_USERNAME = "neo4j"
    NEO4J_PASSWORD = _get_secret_or_prompt("NEO4J_PASSWORD", "Enter NEO4J_PASSWORD: ")
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("Connected to Neo4j Aura.")

if "bedrock_client" not in globals():
    AWS_REGION = input("Enter AWS Region [default: us-east-1]: ").strip() or "us-east-1"
    AWS_ACCESS_KEY_ID = _get_secret_or_prompt("AWS_ACCESS_KEY_ID", "Enter AWS_ACCESS_KEY_ID: ")
    AWS_SECRET_ACCESS_KEY = _get_secret_or_prompt("AWS_SECRET_ACCESS_KEY", "Enter AWS_SECRET_ACCESS_KEY: ")
    BEDROCK_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
    bedrock_client = boto3.client(
        service_name="bedrock-runtime",
        region_name=AWS_REGION,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    print(f"Connected to Amazon Bedrock ({BEDROCK_MODEL_ID}).")

# ==============================================================================
# 2. CURATED HISTORICAL GAZETTEER (BIDAR MONUMENTS & DECCAN REGIONS) + GEOCODER
#    Resolves 1917 spellings & exact monument coordinates around Bidar accurately
# ==============================================================================
HISTORICAL_GAZETTEER = {
    # Local Bidar Monuments & Heritage Sites
    "Bidar Fort":                  {"lat": 17.9219, "lon": 77.5301, "category": "Monument / Fort",       "region": "Bidar, Karnataka"},
    "Bahmani Tombs (Ashtur)":      {"lat": 17.9235, "lon": 77.5588, "category": "Monument / Tomb",       "region": "Ashtur, Bidar"},
    "Barid Shahi Tombs":           {"lat": 17.9168, "lon": 77.5035, "category": "Monument / Tomb",       "region": "Bidar, Karnataka"},
    "Tomb of Ali Barid":           {"lat": 17.9173, "lon": 77.5048, "category": "Monument / Tomb",       "region": "Bidar, Karnataka"},
    "Madrasah of Mahmud Gawan":    {"lat": 17.9146, "lon": 77.5189, "category": "Monument / Madrasah",   "region": "Old City, Bidar"},
    "Rangin Mahall":               {"lat": 17.9203, "lon": 77.5276, "category": "Monument / Palace",     "region": "Bidar Fort, Bidar"},
    "Takht Mahall":                {"lat": 17.9232, "lon": 77.5258, "category": "Monument / Palace",     "region": "Bidar Fort, Bidar"},
    "Sola Khamb Mosque":           {"lat": 17.9225, "lon": 77.5269, "category": "Monument / Mosque",     "region": "Bidar Fort, Bidar"},
    "Gagan Mahall":                {"lat": 17.9212, "lon": 77.5262, "category": "Monument / Palace",     "region": "Bidar Fort, Bidar"},
    "Chini Mahall":                {"lat": 17.9208, "lon": 77.5284, "category": "Monument / Palace",     "region": "Bidar Fort, Bidar"},
    "Chaubara":                    {"lat": 17.9134, "lon": 77.5205, "category": "Monument / Tower",      "region": "Old City, Bidar"},
    "Farh Bagh":                   {"lat": 17.8965, "lon": 77.5320, "category": "Monument / Garden",     "region": "Bidar, Karnataka"},
    "Habshi Kot":                  {"lat": 17.9122, "lon": 77.4980, "category": "Monument / Tomb",       "region": "Bidar, Karnataka"},
    "Jami Masjid (Bidar)":         {"lat": 17.9154, "lon": 77.5218, "category": "Monument / Mosque",     "region": "Old City, Bidar"},
    "Kali Masjid (Bidar)":         {"lat": 17.9118, "lon": 77.5192, "category": "Monument / Mosque",     "region": "Old City, Bidar"},
    # Villages, Suburbs & Localities
    "Ashtur":                      {"lat": 17.9242, "lon": 77.5612, "category": "Village / Locality",    "region": "Bidar Taluk"},
    "Naubad":                      {"lat": 17.9348, "lon": 77.4945, "category": "Village / Locality",    "region": "Bidar Taluk"},
    "Kamthana":                    {"lat": 17.8785, "lon": 77.4412, "category": "Village / Locality",    "region": "Bidar District"},
    "Kalyani (Basavakalyan)":      {"lat": 17.8746, "lon": 76.9488, "category": "Town / Historic Fort",  "region": "Bidar District"},
    # Cities, Districts & States / Historical Regions
    "Bidar":                       {"lat": 17.9149, "lon": 77.5197, "category": "City / District",       "region": "Bidar District, Karnataka"},
    "Gulbarga (Kalaburagi)":       {"lat": 17.3297, "lon": 76.8343, "category": "City / District",       "region": "Karnataka"},
    "Hyderabad":                   {"lat": 17.3850, "lon": 78.4867, "category": "City / State Capital",  "region": "Telangana"},
    "Golconda":                    {"lat": 17.3833, "lon": 78.4011, "category": "Historic Fort / City",  "region": "Hyderabad, Telangana"},
    "Bijapur (Vijayapura)":        {"lat": 16.8302, "lon": 75.7100, "category": "City / District",       "region": "Karnataka"},
    "Ahmadnagar":                  {"lat": 19.0948, "lon": 74.7480, "category": "City / District",       "region": "Maharashtra"},
    "Daulatabad (Devagiri)":       {"lat": 19.9427, "lon": 75.2133, "category": "Historic Fort / Town",  "region": "Maharashtra"},
    "Warangal":                    {"lat": 17.9689, "lon": 79.5941, "category": "City / District",       "region": "Telangana"},
    "Delhi":                       {"lat": 28.6139, "lon": 77.2090, "category": "Imperial Capital",      "region": "North India"},
    "Telangana (Telingana)":       {"lat": 17.8495, "lon": 79.1151, "category": "State / Region",        "region": "Deccan, India"},
    "Deccan":                      {"lat": 17.6000, "lon": 77.2000, "category": "Historical Region",     "region": "Southern India"},
}

ALIAS_MAP = {
    "ahmadabad bidar": "Bidar",
    "bidar town": "Bidar",
    "bidar city": "Bidar",
    "fort of bidar": "Bidar Fort",
    "bahmani tombs": "Bahmani Tombs (Ashtur)",
    "tombs of bahmani kings": "Bahmani Tombs (Ashtur)",
    "tomb of ali barid": "Tomb of Ali Barid",
    "tomb of 'ali barid": "Tomb of Ali Barid",
    "madrasah": "Madrasah of Mahmud Gawan",
    "mahmud gawan's madrasah": "Madrasah of Mahmud Gawan",
    "sola khamb": "Sola Khamb Mosque",
    "solah khamb": "Sola Khamb Mosque",
    "gulbarga": "Gulbarga (Kalaburagi)",
    "bijapur": "Bijapur (Vijayapura)",
    "kalyani": "Kalyani (Basavakalyan)",
    "daulatabad": "Daulatabad (Devagiri)",
    "telangana": "Telangana (Telingana)",
    "telingana": "Telangana (Telingana)",
}

geolocator = Nominatim(user_agent="bidar_graphrag_spatial_explorer_2026")
geocode_fn = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

def normalize_and_geocode(raw_name: str, category_hint: str = "Location"):
    cleaned = raw_name.strip()
    key_lower = cleaned.lower()
    canonical = ALIAS_MAP.get(key_lower, cleaned)

    # 1. Exact match in curated historical gazetteer
    if canonical in HISTORICAL_GAZETTEER:
        meta = HISTORICAL_GAZETTEER[canonical]
        return canonical, meta["lat"], meta["lon"], meta["category"], meta["region"]

    # 2. Partial match against gazetteer keys
    for g_name, meta in HISTORICAL_GAZETTEER.items():
        if key_lower in g_name.lower() or g_name.lower() in key_lower:
            return g_name, meta["lat"], meta["lon"], meta["category"], meta["region"]

    # 3. Fallback to OpenStreetMap Nominatim (scoped to India)
    try:
        loc = geocode_fn(f"{canonical}, India")
        if loc:
            return canonical, round(loc.latitude, 4), round(loc.longitude, 4), category_hint, "India"
    except Exception:
        pass

    return None

# ==============================================================================
# 3. EXTRACT SPATIAL ENTITIES PER CHUNK USING AMAZON BEDROCK
# ==============================================================================
with driver.session() as session:
    chunk_rows = session.run(
        """
        MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)-[:CONTAINS_PAGE]->(p:Page)
        WHERE p.has_text = true AND size(trim(p.text)) > 0
        OPTIONAL MATCH (p)-[:MENTIONS_ENTITY]->(e:ENTITIES)
        OPTIONAL MATCH (p)-[:HAS_THEME]->(th:THEME)
        WITH d, c, p,
             collect(DISTINCT e.name) AS page_years,
             collect(DISTINCT th.name) AS page_themes
        ORDER BY p.page_number ASC
        RETURN
          d.title AS doc_title,
          c.node_id AS chunk_id,
          c.chunk_index AS chunk_index,
          c.start_page AS start_page,
          c.end_page AS end_page,
          collect({
            page: p.page_number,
            text: p.text,
            years: page_years,
            themes: page_themes
          }) AS pages_data
        ORDER BY c.chunk_index ASC
        """
    ).data()

print(f"Scanning {len(chunk_rows)} active chunks for spatial entities (monuments, villages, districts, states)...")

raw_mentions = []

extraction_system = [{
    "text": (
        "You are a spatial-historical entity extractor analyzing 'The Antiquities of Bidar' (1917). "
        "Extract all geographic locations, monuments, forts, tombs, mosques, palaces, villages, towns, districts, and states/regions mentioned in the provided pages.\n"
        "Return STRICT valid JSON only:\n"
        '{"locations": [{"name": "Location Name", "category": "Monument / Fort | Village / Locality | City / District | State / Region", "pages": [10, 12]}]}'
    )
}]

for ch in chunk_rows:
    c_id = ch["chunk_id"]
    pages_formatted = "\n\n".join(
        f"[Page {p['page']} | Years: {', '.join(p['years']) or 'None'} | Themes: {', '.join(p['themes']) or 'None'}]\n{p['text']}"
        for p in ch["pages_data"]
    )

    resp = bedrock_client.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": f"CHUNK: {c_id}\n\n{pages_formatted}"}]}],
        system=extraction_system,
        inferenceConfig={"temperature": 0.1, "maxTokens": 1000},
    )
    raw_txt = resp["output"]["message"]["content"][0]["text"]
    cleaned_json = re.sub(r"^```(?:json)?|```$", "", raw_txt.strip(), flags=re.MULTILINE).strip()

    try:
        parsed_locs = json.loads(cleaned_json).get("locations", [])
    except Exception:
        parsed_locs = []

    valid_chunk_pages = {p["page"] for p in ch["pages_data"]}
    for item in parsed_locs:
        name = item.get("name", "").strip()
        if not name or len(name) < 3:
            continue
        p_list = [int(x) for x in item.get("pages", []) if int(x) in valid_chunk_pages] or sorted(valid_chunk_pages)
        raw_mentions.append({
            "raw_name": name,
            "category": item.get("category", "Location"),
            "chunk_id": c_id,
            "pages": p_list,
        })

# ==============================================================================
# 4. DEDUPLICATE, GEOCODE & AGGREGATE PROVENANCE (PAGES, CHUNKS, YEARS, THEMES)
# ==============================================================================
page_lookup = {}
for ch in chunk_rows:
    for p in ch["pages_data"]:
        page_lookup[p["page"]] = {
            "chunk_id": ch["chunk_id"],
            "text": p["text"],
            "years": p["years"],
            "themes": p["themes"],
        }

spatial_registry = {}

for m in raw_mentions:
    geo = normalize_and_geocode(m["raw_name"], m["category"])
    if not geo:
        continue
    canonical_name, lat, lon, cat, region_label = geo

    if canonical_name not in spatial_registry:
        spatial_registry[canonical_name] = {
            "node_id": "loc_" + re.sub(r"[^a-z0-9]+", "_", canonical_name.lower()).strip("_"),
            "name": canonical_name,
            "category": cat,
            "region": region_label,
            "lat": lat,
            "lon": lon,
            "pages": set(),
            "chunks": set(),
            "years": set(),
            "themes": set(),
        }

    entry = spatial_registry[canonical_name]
    for p_num in m["pages"]:
        if p_num in page_lookup:
            entry["pages"].add(p_num)
            entry["chunks"].add(page_lookup[p_num]["chunk_id"])
            entry["years"].update(page_lookup[p_num]["years"])
            entry["themes"].update(page_lookup[p_num]["themes"])

print(f"Resolved {len(spatial_registry)} unique geocoded spatial locations. Generating clean interdisciplinary summaries...")

# ==============================================================================
# 5. GENERATE CLEAN 300-CHAR FACTS + 3-4 INTERDISCIPLINARY BULLETS PER LOCATION
# ==============================================================================
synthesis_system = [{
    "text": (
        "You are writing historical-spatial briefs for an interdisciplinary team (historians, urban planners, policy analysts, and non-technical researchers) "
        "based strictly on 'The Antiquities of Bidar' (1917) by G. Yazdani.\n"
        "Return STRICT valid JSON:\n"
        "{\n"
        '  "clean_fact_300": "A crisp, jargon-free executive summary of this place strictly UNDER 295 characters.",\n'
        '  "bullets": [\n'
        '    "Bullet 1: Clear historical/architectural fact with date and [Page X | chunk_Y] citation.",\n'
        '    "Bullet 2: ...",\n'
        '    "Bullet 3: ..."\n'
        '  ],\n'
        '  "theme_highlights": {\n'
        '    "THEME_NAME": "One concise sentence connecting this location to that theme [Page X]."\n'
        "  }\n"
        "}\n"
        "Rules:\n"
        "1. clean_fact_300 MUST be <= 300 characters, plain English, zero OCR gibberish.\n"
        "2. Provide 3 to 4 clear, readable bullet points covering what the text says, dates (A.D./A.H.), and page/chunk citations.\n"
        "3. Use ONLY facts present in the provided page excerpts."
    )
}]

doc_title = chunk_rows[0]["doc_title"] if chunk_rows else "1917_The Antoquities of Bidar _book.pdf"
spatial_records = []

for canonical_name, entry in sorted(spatial_registry.items(), key=lambda x: (-len(x[1]["pages"]), x[0])):
    sorted_pages = sorted(entry["pages"])
    sorted_chunks = sorted(entry["chunks"])
    sorted_years = sorted(entry["years"])
    sorted_themes = sorted(entry["themes"])

    # Gather up to 4 most relevant page texts for synthesis
    excerpts = "\n\n".join(
        f"--- [Page {p} | {page_lookup[p]['chunk_id']}] ---\n{page_lookup[p]['text'][:1800]}"
        for p in sorted_pages[:4]
    )

    user_msg = (
        f"LOCATION: {canonical_name} ({entry['category']}, {entry['region']})\n"
        f"PAGES: {sorted_pages} | CHUNKS: {sorted_chunks}\n"
        f"LINKED YEARS: {', '.join(sorted_years) or 'None'}\n"
        f"LINKED THEMES: {', '.join(sorted_themes) or 'General Heritage & Architecture'}\n\n"
        f"BOOK EXCERPTS:\n{excerpts}"
    )

    resp = bedrock_client.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": user_msg}]}],
        system=synthesis_system,
        inferenceConfig={"temperature": 0.15, "maxTokens": 700},
    )
    raw_out = resp["output"]["message"]["content"][0]["text"]
    cleaned_out = re.sub(r"^```(?:json)?|```$", "", raw_out.strip(), flags=re.MULTILINE).strip()

    try:
        synth = json.loads(cleaned_out)
    except Exception:
        synth = {
            "clean_fact_300": f"{canonical_name} is documented across pages {sorted_pages} in The Antiquities of Bidar (1917).",
            "bullets": [f"Referenced on pages {sorted_pages} ({', '.join(sorted_chunks)}) in G. Yazdani's 1917 survey."],
            "theme_highlights": {},
        }

    # Enforce strict <= 300 char ceiling on clean_fact_300
    fact_300 = synth.get("clean_fact_300", "").strip()
    if len(fact_300) > 300:
        fact_300 = fact_300[:297].rstrip() + "..."

    bullets = synth.get("bullets", [])[:4]
    theme_highlights = synth.get("theme_highlights", {})

    citation_str = (
        f"Yazdani, G. (1917). The Antiquities of Bidar. "
        f"pp. {', '.join(map(str, sorted_pages))} ({', '.join(sorted_chunks)})."
    )

    spatial_records.append({
        "node_id": entry["node_id"],
        "name": canonical_name,
        "category": entry["category"],
        "region": entry["region"],
        "lat": entry["lat"],
        "lon": entry["lon"],
        "pages": sorted_pages,
        "chunks": sorted_chunks,
        "years": sorted_years,
        "themes": sorted_themes,
        "clean_fact_300": fact_300,
        "bullets": bullets,
        "theme_highlights_json": json.dumps(theme_highlights, ensure_ascii=False),
        "citation": citation_str,
        "doc_title": doc_title,
    })

# ==============================================================================
# 6. STORE :Location NODES & SPATIAL RELATIONSHIPS IN NEO4J AURA
# ==============================================================================
with driver.session() as session:
    session.run(
        "CREATE CONSTRAINT location_id_unique IF NOT EXISTS FOR (l:Location) REQUIRE l.node_id IS UNIQUE"
    )
    session.run("MATCH (l:Location) DETACH DELETE l")

    session.run(
        """
        UNWIND $rows AS row
        MERGE (l:Location {node_id: row.node_id})
        SET l.name = row.name,
            l.category = row.category,
            l.region = row.region,
            l.lat = row.lat,
            l.lon = row.lon,
            l.pages = row.pages,
            l.chunks = row.chunks,
            l.years = row.years,
            l.themes = row.themes,
            l.clean_fact_300 = row.clean_fact_300,
            l.bullets = row.bullets,
            l.theme_highlights_json = row.theme_highlights_json,
            l.citation = row.citation
        """,
        rows=spatial_records,
    )

    # Link (:Page)-[:MENTIONS_LOCATION]->(:Location)
    session.run(
        """
        MATCH (l:Location)
        UNWIND l.pages AS p_num
        MATCH (p:Page {page_number: p_num})
        MERGE (p)-[:MENTIONS_LOCATION]->(l)
        """
    )

    # Link (:Location)-[:LOCATED_IN_CHUNK]->(:Chunk)
    session.run(
        """
        MATCH (l:Location)
        UNWIND l.chunks AS c_id
        MATCH (c:Chunk {node_id: c_id})
        MERGE (l)-[:LOCATED_IN_CHUNK]->(c)
        """
    )

    # Link (:Location)-[:ACTIVE_IN_YEAR]->(:ENTITIES)
    session.run(
        """
        MATCH (l:Location)
        UNWIND l.years AS y_name
        MATCH (e:ENTITIES {name: y_name})
        MERGE (l)-[:ACTIVE_IN_YEAR]->(e)
        """
    )

    # Link (:Location)-[:HAS_THEME]->(:THEME)
    session.run(
        """
        MATCH (l:Location)
        UNWIND l.themes AS t_name
        MATCH (th:THEME {name: t_name})
        MERGE (l)-[:HAS_THEME]->(th)
        """
    )

    # Create Spatial Co-Occurrence Network Edges (:Location)-[:CONNECTED_TO]->(:Location)
    session.run(
        """
        MATCH (l1:Location)<-[:MENTIONS_LOCATION]-(p:Page)-[:MENTIONS_LOCATION]->(l2:Location)
        WHERE l1.node_id < l2.node_id
        WITH l1, l2, collect(DISTINCT p.page_number) AS shared_pages, collect(DISTINCT p.chunk_id) AS shared_chunks
        MERGE (l1)-[rel:CONNECTED_TO]-(l2)
        SET rel.shared_pages = shared_pages,
            rel.shared_chunks = shared_chunks,
            rel.strength = size(shared_pages)
        """
    )

print(f"Stored {len(spatial_records)} :Location nodes and spatial cross-chunk relationships in Neo4j Aura!")

Scanning 13 active chunks for spatial entities (monuments, villages, districts, states)...
Resolved 57 unique geocoded spatial locations. Generating clean interdisciplinary summaries...
Stored 57 :Location nodes and spatial cross-chunk relationships in Neo4j Aura!


In [22]:
!pip install -q folium

import html as html_lib
import json
import folium
from folium.plugins import Fullscreen
import pandas as pd
from getpass import getpass
from neo4j import GraphDatabase
from IPython.display import HTML, Markdown, display

# 1. Reconnect to Neo4j Aura if needed (no Bedrock re-extraction required)
if "driver" not in globals():
    NEO4J_URI = "neo4j+s://c7554cbf.databases.neo4j.io"
    NEO4J_USERNAME = "neo4j"
    try:
        from google.colab import userdata
        NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")
    except Exception:
        NEO4J_PASSWORD = None
    if not NEO4J_PASSWORD:
        NEO4J_PASSWORD = getpass("Enter NEO4J_PASSWORD: ").strip()
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()

# 2. Fetch Stored Spatial Nodes & Cross-Location Graph Edges from Neo4j Aura
with driver.session() as session:
    locations = session.run(
        """
        MATCH (l:Location)
        OPTIONAL MATCH (l)-[r:CONNECTED_TO]-(other:Location)
        WITH l,
             collect(DISTINCT {
               name: other.name,
               lat: other.lat,
               lon: other.lon,
               shared_pages: r.shared_pages,
               shared_chunks: r.shared_chunks
             }) AS connected_places
        RETURN
          l.node_id AS node_id,
          l.name AS name,
          l.category AS category,
          l.region AS region,
          l.lat AS lat,
          l.lon AS lon,
          l.pages AS pages,
          l.chunks AS chunks,
          l.years AS years,
          l.themes AS themes,
          l.clean_fact_300 AS clean_fact_300,
          l.bullets AS bullets,
          l.theme_highlights_json AS theme_highlights_json,
          l.citation AS citation,
          connected_places
        ORDER BY size(l.pages) DESC, l.name ASC
        """
    ).data()

# 3. Initialize Clean Map with Keyless Esri Basemaps (Works inside Colab Iframes)
m = folium.Map(
    location=[17.9175, 77.5265],  # Centered on Bidar Fort, Old City & Ashtur Tombs
    zoom_start=13,
    tiles=None,
    control_scale=True,
)

# Primary Basemap: Warm beige/tan street & terrain style matching your reference image
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ, USGS, Intermap, TomTom",
    name="Heritage Street Map (Default)",
    control=True,
).add_to(m)

# Optional Alternate Basemaps in Top-Right Layer Menu
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Esri, USGS, NOAA",
    name="Topographic Terrain",
    control=True,
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Source: Esri, Maxar, Earthstar Geographics",
    name="Satellite Imagery",
    control=True,
).add_to(m)

Fullscreen(position="topleft").add_to(m)

# 4. Feature Groups (Cross-Chunk Spiderweb Lines OFF by default for a clean view)
fg_monuments = folium.FeatureGroup(name="Bidar Monuments & Heritage Sites", show=True)
fg_localities = folium.FeatureGroup(name="Villages & Localities", show=True)
fg_regional = folium.FeatureGroup(name="Cities, Districts & Regions", show=True)
fg_links = folium.FeatureGroup(name="Cross-Chunk Spatial Links (Toggle On)", show=False)

THEME_BADGE_COLORS = {
    "Location & Administrative Units": "#d97706",
    "Government & Policies":           "#7c3aed",
    "Irrigation & Water":              "#0284c7",
    "Agriculture & Land Use":          "#15803d",
    "Climate & Heat":                  "#ea580c",
    "Socio-Economic Inequality":       "#b45309",
    "Health & Sanitation":             "#dc2626",
}

def get_pin_color(cat: str) -> str:
    c = cat.lower()
    if "monument" in c or "fort" in c or "tomb" in c or "mosque" in c or "palace" in c or "historic" in c:
        return "#1e293b"  # Dark Slate Circle (matches reference screenshot)
    if "village" in c or "town" in c or "locality" in c:
        return "#0f766e"  # Deep Teal Circle
    return "#1d4ed8"      # Royal Blue Circle for Cities/Districts/States

# 5. Add Optional Cross-Chunk Relationship Polylines (Toggleable)
drawn_pairs = set()
for loc in locations:
    for conn in loc["connected_places"]:
        if not conn.get("name") or conn.get("lat") is None:
            continue
        pair_key = tuple(sorted([loc["name"], conn["name"]]))
        if pair_key in drawn_pairs:
            continue
        drawn_pairs.add(pair_key)

        shared_p = conn.get("shared_pages") or []
        shared_c = conn.get("shared_chunks") or []
        folium.PolyLine(
            locations=[[loc["lat"], loc["lon"]], [conn["lat"], conn["lon"]]],
            color="#475569",
            weight=1.5,
            opacity=0.45,
            dash_array="5, 6",
            tooltip=(
                f"<b>Spatial Link:</b> {loc['name']} ↔ {conn['name']}<br>"
                f"<b>Shared Pages:</b> {shared_p} | <b>Chunks:</b> {', '.join(shared_c)}"
            ),
        ).add_to(fg_links)

# 6. Add Sleek Circular Heritage Pins + Clean White Popup Cards
for loc in locations:
    theme_highlights = {}
    try:
        theme_highlights = json.loads(loc.get("theme_highlights_json") or "{}")
    except Exception:
        theme_highlights = {}

    years_str = ", ".join(loc["years"]) if loc["years"] else "Historical Period (Undated on Page)"
    pages_str = ", ".join(f"p.{p}" for p in loc["pages"])
    chunks_str = ", ".join(loc["chunks"])
    connected_names = [c["name"] for c in loc["connected_places"] if c.get("name")]
    connected_str = ", ".join(connected_names[:6]) if connected_names else "Standalone reference"

    theme_sections_html = ""
    if theme_highlights:
        for t_name, t_note in theme_highlights.items():
            badge_col = THEME_BADGE_COLORS.get(t_name, "#b45309")
            theme_sections_html += f"""
            <div style="margin-top:8px;">
              <div style="font-size:10px; font-weight:700; letter-spacing:0.6px; color:{badge_col}; text-transform:uppercase;">
                {html_lib.escape(t_name)}
              </div>
              <div style="font-size:12px; color:#334155; margin-top:2px; line-height:1.4;">
                • {html_lib.escape(str(t_note))}
              </div>
            </div>
            """

    bullets_html = "".join(
        f"<li style='margin-bottom:4px;'>{html_lib.escape(b)}</li>"
        for b in (loc["bullets"] or [])
    )

    popup_html = f"""
    <div style="font-family:-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; width:355px; max-height:390px; overflow-y:auto; color:#1e293b; padding-right:4px;">
      <div style="border-bottom:1px solid #e2e8f0; padding-bottom:8px;">
        <div style="font-size:16px; font-weight:700; color:#0f172a;">📍 {html_lib.escape(loc['name'])}</div>
        <div style="font-size:11px; color:#64748b; margin-top:1px;">{html_lib.escape(loc['category'])} • {html_lib.escape(loc['region'])}</div>
      </div>

      <div style="font-size:11.5px; color:#64748b; font-style:italic; margin:6px 0 8px 0;">
        Explore below the historical page context, timeline, and themes found in this location:
      </div>

      <div style="background:#f8fafc; border-left:3px solid #0284c7; padding:8px 10px; border-radius:4px; font-size:12.5px; color:#0f172a; line-height:1.45;">
        {html_lib.escape(loc['clean_fact_300'])}
      </div>

      <div style="margin-top:8px; font-size:11.5px; background:#f1f5f9; padding:6px 8px; border-radius:4px; color:#334155;">
        <div><b>🕒 Timeline:</b> {html_lib.escape(years_str)}</div>
        <div style="margin-top:2px;"><b>📄 Provenance:</b> {html_lib.escape(pages_str)} ({html_lib.escape(chunks_str)})</div>
      </div>

      {theme_sections_html}

      <div style="margin-top:9px;">
        <div style="font-size:10px; font-weight:700; letter-spacing:0.6px; color:#0f766e; text-transform:uppercase;">
          Historical & Architectural Context
        </div>
        <ul style="margin:4px 0 0 16px; padding:0; font-size:12px; color:#334155; line-height:1.42;">
          {bullets_html}
        </ul>
      </div>

      <div style="margin-top:9px; padding-top:7px; border-top:1px dashed #cbd5e1; font-size:11px; color:#475569;">
        <div><b>🔗 Connected Locations:</b> {html_lib.escape(connected_str)}</div>
        <div style="margin-top:3px; color:#64748b;"><b>📚 Citation:</b> {html_lib.escape(loc['citation'])}</div>
      </div>
    </div>
    """

    pin_color = get_pin_color(loc["category"])
    custom_circle_icon = folium.DivIcon(
        icon_size=(24, 24),
        icon_anchor=(12, 12),
        popup_anchor=(0, -12),
        html=f"""
        <div style="
            width: 22px; height: 22px;
            background-color: {pin_color};
            border: 2.5px solid #ffffff;
            border-radius: 50%;
            box-shadow: 0 2px 6px rgba(15, 23, 42, 0.45);
            display: flex; align-items: center; justify-content: center;
            cursor: pointer;
        ">
            <div style="width: 6px; height: 6px; background-color: #ffffff; border-radius: 50%;"></div>
        </div>
        """,
    )

    marker = folium.Marker(
        location=[loc["lat"], loc["lon"]],
        tooltip=f"<b>{loc['name']}</b> ({loc['category']}) — Click to inspect facts & themes",
        popup=folium.Popup(popup_html, max_width=380),
        icon=custom_circle_icon,
    )

    cat_lower = loc["category"].lower()
    if any(k in cat_lower for k in ["monument", "fort", "tomb", "mosque", "palace", "historic"]):
        marker.add_to(fg_monuments)
    elif any(k in cat_lower for k in ["village", "town", "locality"]):
        marker.add_to(fg_localities)
    else:
        marker.add_to(fg_regional)

fg_links.add_to(m)
fg_monuments.add_to(m)
fg_localities.add_to(m)
fg_regional.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Save updated standalone HTML file
map_html_file = "bidar_spatial_heritage_map.html"
m.save(map_html_file)

display(Markdown(
    "## Interactive Spatial Heritage Map — *The Antiquities of Bidar* (1917)\n"
    "`● Dark Slate = Monuments & Forts` | `● Teal = Villages & Localities` | `● Blue = Cities, Districts & Regions`  \n"
    "*Click any circular pin to view its clean 300-char brief, timeline, themes, and citations. Use the top-right layer menu to switch basemaps or toggle cross-chunk connection lines.*"
))

map_raw_html = m.get_root().render()
iframe_map = (
    f'<iframe srcdoc="{html_lib.escape(map_raw_html, quote=True)}" '
    f'style="width:100%; height:620px; border:1px solid #cbd5e1; border-radius:10px; box-shadow:0 4px 16px rgba(0,0,0,0.08);">'
    f'</iframe>'
)
display(HTML(iframe_map))

## Interactive Spatial Heritage Map — *The Antiquities of Bidar* (1917)
`● Dark Slate = Monuments & Forts` | `● Teal = Villages & Localities` | `● Blue = Cities, Districts & Regions`  
*Click any circular pin to view its clean 300-char brief, timeline, themes, and citations. Use the top-right layer menu to switch basemaps or toggle cross-chunk connection lines.*

/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
